# Hiroyoshi Qwen3 with supervised MathOctopus distillation

The Hiroyoshi `NonlinearMultiply` block construction is kept. Each merge still updates
only the new merged block using hidden-state matching and teacher-logit distillation.
MathOctopus distillation windows also train the block on the assistant's worked solution
and canonical `Final answer:` from the matched GSM8K training problem. The response loss
has a weight of 0.25 and is included in epoch selection on the separate training holdout.
Wiki-40B windows continue to use the original losses.

The checkpoint, layer pairs, epochs, source mix, token limits, evaluation seeds, prompts,
and scorers follow notebook (15). The training examples remain subject to the same
256-token math window cap. The added answer formatting can change which examples fit.
The test scores and outputs are generated anew when you run the notebook.

Run all cells with Kaggle GPU and Internet enabled. The revised training code is embedded
in this notebook and written to its own working folder. Keep `SAVED_RUN_DIR = None` to
train with the revised objective. Save the results ZIP after the run if you want to
reuse the checkpoints later.


In [6]:
import time

start_time=time.time()

In [7]:
from pathlib import Path

SAVED_RUN_DIR = None  # Or an extracted results directory to reuse its checkpoints.
AUDIT_RUN_DIR = None  # Or an extracted prior run to audit and exclude.
EVAL_SEED = 43  # Independent of the training seed and prior test order.

MODEL_ID = 'Qwen/Qwen3-0.6B'
FIXED_PAIRS = ('7-8', '13-14')  # One-based original layer numbers.
THIRD_PAIR_CANDIDATES = ()
MAX_THIRD_PPL_DEGRADATION_PCT = 5.0  # Relative to Original on this run's validation set.
TRAIN_EPOCHS = 12
MATH_TRAIN_FRACTION = 0.5  # Fraction of windows in each training and holdout subset.
MATH_TRAIN_LANGUAGES = ('de', 'ja', 'fr')
MATH_MAX_WINDOW_TOKENS = 256  # Keep complete math question-and-solution windows.
EVAL_LANGUAGES = ('de', 'ja', 'fr')
QUESTIONS_PER_BENCHMARK_LANGUAGE = 40
DEMONSTRATIONS_PER_QUESTION = 5
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 256  # MGSM only.
RUN_MCQ_GENERATION_DIAGNOSTIC = False  # Recognizes answers such as "The correct answer is: C".
MCQ_GENERATION_MAX_NEW_TOKENS = 64  # Longer output costs more GPU time.
TIME_LIMIT_SECONDS = None  # Set to 20 * 60 for a hard cutoff on Kaggle.

BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORK = BASE / 'hiroyoshi_qwen3_mathoctopus_supervised'
WORK.mkdir(parents=True, exist_ok=True)

In [8]:
from hashlib import sha256
SOURCES = {'Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""Run Hiroyoshi\'s Qwen3 layer-merging pilot in a Kaggle GPU session.\n\nThe pilot defaults to NonlinearMultiply on original layers 7-8 and 13-14.\nIt reports Wiki-40B validation perplexity after every stage and compares the\noriginal and final models on small held-out samples of MMMLU, INCLUDE and MGSM.\nUse --mode quick or --mode full for the original greedy search across all pairs.\n\nExamples in an environment with the pinned dependencies and a GPU:\n  python Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py\n  python Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py --checks-only\n  python Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py --mode full --max-merges 5\n"""\n\nfrom __future__ import annotations\nimport argparse\nimport importlib\nimport importlib.metadata\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nDEPENDENCIES = (\n    "transformers==4.57.6", "datasets==4.4.1", "numpy==2.2.6", "scipy==1.16.3",\n    "pandas==2.2.3", "threadpoolctl==3.6.0", "accelerate==1.12.0",\n    "fsspec==2025.10.0",\n)\n\n# Embedded experiment modules extracted into the run workspace.\nMODULE_SOURCES = {\'qwen3_affine_core.py\': \'\\n"""Affine/evaluation utilities adapted from the supplied notebook.\\nOnly functions needed by Nonlinear Composition Merging are included.\\n"""\\nfrom __future__ import annotations\\nimport os\\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\\nos.environ.setdefault("USE_TF", "0")\\nos.environ.setdefault("USE_FLAX", "0")\\nimport math\\nimport random\\nimport time\\nimport numpy as np\\nimport scipy\\nfrom scipy.linalg import cho_factor, cho_solve\\nimport torch\\nfrom torch import nn\\nimport transformers\\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\\n\\ndef set_seed(seed):\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n    torch.use_deterministic_algorithms(True)\\n    torch.backends.cudnn.benchmark = False\\n    torch.backends.cudnn.deterministic = True\\n    torch.backends.cuda.matmul.allow_tf32 = False\\n    torch.backends.cudnn.allow_tf32 = False\\n\\n\\ndef _array(x):\\n    if torch.is_tensor(x):\\n        x = x.detach().cpu().numpy()\\n    a = np.asarray(x, dtype=np.float64)\\n    if a.ndim != 2 or not np.isfinite(a).all():\\n        raise ValueError("Expected a finite [tokens, features] matrix")\\n    return a\\n\\n\\ndef _relative_reg(gram, strength):\\n    if strength <= 0:\\n        raise ValueError("A positive ridge is required")\\n    scale = float(np.trace(gram) / gram.shape[0])\\n    return strength * max(scale, np.finfo(np.float64).eps)\\n\\n\\ndef fit_affine(X, Y, ridge=1e-4):\\n    """min ||XA+b-Y||_F^2 + lambda ||A||_F^2; bias unpenalized.\\n\\n    lambda = ridge * trace(Xc.T Xc)/d, matching the previous experiment.\\n    No explicit inverse; CPU float64 is intentional for stable CCA/ridge fits.\\n    """\\n    X, Y = _array(X), _array(Y)\\n    if len(X) != len(Y) or len(X) < 2:\\n        raise ValueError("X and Y need the same N >= 2")\\n    mx, my = X.mean(0), Y.mean(0)\\n    xc, yc = X - mx, Y - my\\n    gram = xc.T @ xc\\n    reg = _relative_reg(gram, ridge)\\n    A = cho_solve(cho_factor(gram + reg * np.eye(X.shape[1])), xc.T @ yc)\\n    b = my - mx @ A\\n    return A, b\\n\\n\\nclass AffineMergedBlock(nn.Module):\\n    """Qwen3 decoder-layer-compatible affine full-span approximation.\\n\\n    Qwen3DecoderLayer returns a Tensor (GPT-2 returned a tuple). This block is\\n    valid for use_cache=False (as used throughout this experiment).\\n    """\\n    def __init__(self, A, b, device=None, dtype=torch.float32,\\n                 attention_type="full_attention"):\\n        super().__init__()\\n        # transformers==4.57.6 Qwen3Model.forward selects the causal mask by\\n        # decoder_layer.attention_type, BEFORE it calls decoder_layer.forward.\\n        # A replacement block must expose that attribute even when it has\\n        # no self-attention module of its own.\\n        if attention_type not in ("full_attention", "sliding_attention"):\\n            raise ValueError(f"Unsupported Qwen3 attention_type: {attention_type}")\\n        self.attention_type = attention_type\\n        d = A.shape[0]\\n        self.linear = nn.Linear(d, d, device=device, dtype=dtype)\\n        with torch.no_grad():\\n            self.linear.weight.copy_(torch.as_tensor(np.array(A.T, copy=True)))\\n            self.linear.bias.copy_(torch.as_tensor(np.array(b, copy=True)))\\n\\n    def forward(self, hidden_states, attention_mask=None, position_ids=None,\\n                past_key_values=None, use_cache=False, cache_position=None,\\n                position_embeddings=None, **kwargs):\\n        if use_cache or past_key_values is not None:\\n            raise ValueError("Affine block only supports full-sequence use_cache=False")\\n        return self.linear(hidden_states)\\n\\n\\ndef parameter_count(model):\\n    return sum(p.numel() for p in model.parameters())  # tied weights counted once\\n\\n\\ndef make_state_windows(ids, total_tokens, chunk_size, seed=None):\\n    """Disjoint source windows; random order for TRAIN, sequential for VAL."""\\n    ids = ids.detach().cpu().flatten()\\n    if len(ids) < total_tokens or total_tokens < 2 or chunk_size < 2:\\n        raise ValueError("Invalid number of state tokens / chunk size")\\n    starts = np.arange(0, len(ids), chunk_size)\\n    if seed is not None:\\n        np.random.default_rng(seed).shuffle(starts)\\n    windows, remaining = [], total_tokens\\n    for start in starts:\\n        end = min(int(start) + chunk_size, len(ids), int(start) + remaining)\\n        if end <= start:\\n            continue\\n        windows.append(ids[int(start):end].clone())\\n        remaining -= end - int(start)\\n        if remaining == 0:\\n            break\\n    if remaining:\\n        raise ValueError("Insufficient tokens")\\n    return windows\\n\\n\\n@torch.inference_mode()\\ndef collect_pair_states(model, windows, L):\\n    """Hooks capture pre-ln_f block outputs, including pair (10,11)."""\\n    if model.training:\\n        raise ValueError("Collect states with model.eval()")\\n    d = model.config.hidden_size\\n    n = sum(w.numel() for w in windows)\\n    states = [np.empty((n, d), np.float32) for _ in range(3)]\\n    position = [0, 0, 0]\\n    def store(index, tensor):\\n        a = tensor.detach().reshape(-1, d).float().cpu().numpy()\\n        start = position[index]\\n        states[index][start:start + len(a)] = a\\n        position[index] += len(a)\\n    handles = [\\n        model.model.layers[L].register_forward_pre_hook(lambda m, inp: store(0, inp[0])),\\n        model.model.layers[L].register_forward_hook(lambda m, inp, out: store(1, out)),\\n        model.model.layers[L + 1].register_forward_hook(lambda m, inp, out: store(2, out)),\\n    ]\\n    try:\\n        device = next(model.parameters()).device\\n        for window in windows:\\n            # No vocabulary logits needed while collecting states.\\n            model.model(window[None].to(device), use_cache=False, return_dict=True)\\n    finally:\\n        for handle in handles:\\n            handle.remove()\\n    if position != [n, n, n]:\\n        raise RuntimeError(f"Incomplete activation collection: {position}")\\n    return tuple(states)\\n\\n\\n@torch.inference_mode()\\ndef measure_latency(model, input_ids, repeats=10, warmup=5):\\n    """Batch=1 full LM forward, no KV cache, excludes CPU->GPU transfer."""\\n    if repeats < 1:\\n        raise ValueError("Latency repeats must be positive")\\n    device = next(model.parameters()).device\\n    x = input_ids.flatten()[None].to(device)\\n    def sync():\\n        if device.type == "cuda":\\n            torch.cuda.synchronize(device)\\n    model.eval()\\n    for _ in range(warmup):\\n        model(x, use_cache=False)\\n    sync()\\n    samples = []\\n    for _ in range(repeats):\\n        start = time.perf_counter()\\n        model(x, use_cache=False)\\n        sync()\\n        samples.append(1000 * (time.perf_counter() - start))\\n    return {"Latency_ms": float(np.median(samples)),\\n            "Latency_Mean_ms": float(np.mean(samples)),\\n            "Latency_P90_ms": float(np.quantile(samples, 0.9))}\\n\\n\\ndef check_transformers_version():\\n    actual = transformers.__version__\\n    expected = "4.57.6"\\n    if actual != expected:\\n        raise RuntimeError(\\n            f"Incompatible transformers loaded: {actual} (required: {expected}). "\\n            "Run `%pip install -q transformers==4.57.6` in Kaggle, "\\n            "restart the notebook session, and run again. Installing after "\\n            "transformers was imported cannot replace the loaded module. "\\n            "Do not disable this check: the Qwen3 merging wrapper uses "\\n            "the 4.57.6 Qwen3 decoder interface."\\n        )\\n\', \'evaluation_protocol.py\': \'\\n"""Exact sliding/legacy evaluator from merging_comparison_colab.ipynb."""\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport numpy as np\\nimport torch\\n\\ndef target_mask(length, context, protocol):\\n    """Global positions whose next-token losses belong in the denominator."""\\n    if context < 2 or length < 2:\\n        raise ValueError("Need context and token count >= 2")\\n    mask = torch.zeros(length, dtype=torch.bool)\\n    if protocol == "sliding":\\n        mask[1:] = True\\n    elif protocol == "legacy":\\n        for start in range(0, (length//context)*context, context):\\n            mask[start+1:start+context] = True\\n    else:\\n        raise ValueError("protocol must be sliding or legacy")\\n    return mask\\n\\n\\ndef protocol_windows(ids, context, stride, protocol="sliding", score_mask=None):\\n    ids = ids.detach().cpu().flatten()\\n    targets = target_mask(len(ids), context, protocol)\\n    if score_mask is not None:\\n        if score_mask.shape != targets.shape:\\n            raise ValueError("score_mask must match the token sequence")\\n        targets &= score_mask.cpu().bool()\\n    if not targets.any():\\n        raise ValueError("No scored tokens; legacy requires a full context window")\\n    if protocol == "sliding":\\n        if not 1 <= stride < context:\\n            raise ValueError("Sliding evaluation needs 1 <= stride < context")\\n        previous_end = 0\\n        for start in range(0, len(ids), stride):\\n            end = min(start+context, len(ids))\\n            x = ids[start:end][None]\\n            labels = x.clone()\\n            if previous_end > start:\\n                labels[:, :previous_end-start] = -100\\n            labels[:, ~targets[start:end]] = -100\\n            count = int((labels[:, 1:] != -100).sum())\\n            if count:\\n                yield x, labels, count\\n            previous_end = end\\n            if end == len(ids):\\n                break\\n    else:\\n        for start in range(0, (len(ids)//context)*context, context):\\n            x = ids[start:start+context][None]\\n            labels = x.clone()\\n            labels[:, ~targets[start:start+context]] = -100\\n            count = int((labels[:, 1:] != -100).sum())\\n            if count:\\n                yield x, labels, count\\n\\n\\n@torch.inference_mode()\\ndef calculate_ppl(model, ids, context=512, stride=256, protocol="sliding",\\n                  score_mask=None, batch_size=8):\\n    """Token-weighted PPL with batched evaluation windows.\\n\\n    This preserves the same sliding-window target mask and denominator as the\\n    original evaluator. Windows of equal length are grouped into a GPU batch,\\n    reducing Python/kernel-launch overhead without changing which tokens score.\\n    """\\n    if batch_size < 1:\\n        raise ValueError("batch_size must be >= 1")\\n\\n    model.eval()\\n    device = next(model.parameters()).device\\n    total_nll, total_count = 0.0, 0\\n\\n    pending_x, pending_labels, pending_counts = [], [], []\\n    pending_len = None\\n\\n    def flush():\\n        nonlocal total_nll, total_count, pending_x, pending_labels\\n        nonlocal pending_counts, pending_len\\n        if not pending_x:\\n            return\\n\\n        x_batch = torch.cat(pending_x, dim=0).to(device)\\n        labels_batch = torch.cat(pending_labels, dim=0).to(device)\\n        count = int(sum(pending_counts))\\n\\n        output = model(\\n            input_ids=x_batch,\\n            labels=labels_batch,\\n            use_cache=False,\\n            return_dict=True,\\n        )\\n        loss = float(output.loss)\\n        if not math.isfinite(loss):\\n            raise ArithmeticError("Non-finite PPL loss")\\n\\n        total_nll += loss * count\\n        total_count += count\\n\\n        pending_x, pending_labels, pending_counts = [], [], []\\n        pending_len = None\\n\\n    for x, labels, count in protocol_windows(ids, context, stride, protocol, score_mask):\\n        seq_len = x.shape[1]\\n\\n        # The final sliding window can be shorter; do not pad because that would\\n        # unnecessarily alter the exact input sequence used by the old evaluator.\\n        if pending_x and (seq_len != pending_len or len(pending_x) >= batch_size):\\n            flush()\\n\\n        pending_x.append(x)\\n        pending_labels.append(labels)\\n        pending_counts.append(count)\\n        pending_len = seq_len\\n\\n        if len(pending_x) >= batch_size:\\n            flush()\\n\\n    flush()\\n\\n    expected = target_mask(ids.numel(), context, protocol)\\n    if score_mask is not None:\\n        expected &= score_mask.cpu().bool()\\n    if total_count != int(expected.sum()):\\n        raise AssertionError("Incorrect next-token denominator")\\n\\n    nll = total_nll / total_count\\n    return {\\n        "PPL": math.exp(nll) if nll < 700 else float("inf"),\\n        "NLL": nll,\\n        "Predicted_Tokens": total_count,\\n    }\\n\\n\\ndef _write_json(path, value):\\n    def normalize(x):\\n        if isinstance(x, dict):\\n            return {k: normalize(v) for k, v in x.items()}\\n        if isinstance(x, (list, tuple)):\\n            return [normalize(v) for v in x]\\n        if isinstance(x, np.generic):\\n            return normalize(x.item())\\n        if isinstance(x, float) and not math.isfinite(x):\\n            return None\\n        return x\\n    Path(path).write_text(json.dumps(normalize(value), ensure_ascii=False, indent=2,\\n                                    allow_nan=False), encoding="utf-8")\\n\', \'qwen3_compression_multi.py\': \'\\n"""Greedy adjacent-layer compression of Qwen3 (adapted from the uploaded GPT-2 experiment).\\n\\nMethods: LinearMultiply, Remove, Average, NonlinearMultiply. Training uses\\nWiki-40B and optional supervised MathOctopus responses. Wiki-40B is used\\nfor validation. Held-out tests: MMMLU, INCLUDE, MGSM.\\nNo benchmark test label is used for training, selection or few-shot exemplars.\\n"""\\nfrom __future__ import annotations\\n\\nimport copy\\nimport gc\\nimport hashlib\\nimport json\\nimport math\\nimport platform\\nfrom contextlib import contextmanager\\nfrom dataclasses import asdict, dataclass, replace\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\nfrom torch.utils.checkpoint import checkpoint\\nfrom threadpoolctl import threadpool_limits\\n\\nimport qwen3_affine_core as sm\\nimport evaluation_protocol as ep\\n\\nMETHODS = ("LinearMultiply", "Remove", "Average", "NonlinearMultiply")\\nMATH_RESPONSE_WEIGHT = 0.25\\n\\n\\n@dataclass(frozen=True)\\nclass Config:\\n    model_id: str = "Qwen/Qwen3-0.6B"\\n    model_revision: str = "main"\\n    # Six languages shared by Wiki-40B, MMMLU, INCLUDE and MGSM.\\n    train_languages: tuple[str, ...] = ("de", "es", "fr", "ja", "ru", "zh-cn")\\n    wiki_train_tokens_per_language: int = 2500\\n    wiki_validation_tokens_per_language: int = 1200\\n    dataset_revision: str = "main"\\n    seed: int = 42\\n    mode: str = "quick"\\n    context_length: int = 128\\n    eval_stride: int = 64\\n    eval_batch_size: int = 2\\n    eval_tokens: int | None = 10000\\n    hidden_eval_tokens: int = 4096\\n    local_tokens: int = 2048\\n    kd_tokens: int = 2048\\n    holdout_tokens: int = 1024\\n    epochs: int = 10\\n    patience: int = 15\\n    grad_accumulation: int = 1\\n    lr_transformer: float = 3e-5\\n    weight_decay: float = 0.01\\n    max_grad_norm: float = 1.0\\n    lambda_hidden: float = 1.0\\n    lambda_kl: float = 1.0\\n    temperature: float = 2.0\\n    epsilon: float = 1e-8\\n    affine_ridge: float = 1e-4\\n    kl_chunk_tokens: int = 16\\n    cpu_threads: int = 2\\n    latency_seq_len: int = 256\\n    latency_repeats: int = 10\\n    max_merges: int = 5\\n    output_dir: str = "/kaggle/working/qwen3_wiki40b_multi_results"\\n    fixed_layer_pairs: tuple[tuple[int, int], ...] = ()  # One-based original layers.\\n    third_pair_candidates: tuple[tuple[int, int], ...] = ()\\n    max_third_ppl_degradation_pct: float = 5.0\\n    math_train_fraction: float = 0.0\\n    math_train_languages: tuple[str, ...] = ("de", "ja", "fr")\\n    math_max_window_tokens: int = 256\\n\\n\\ndef make_config(mode="quick", **kwargs):\\n    if mode == "pilot":\\n        pilot = replace(\\n            Config(), mode="pilot", fixed_layer_pairs=((7, 8), (13, 14)),\\n            max_merges=2, wiki_train_tokens_per_language=2048,\\n            wiki_validation_tokens_per_language=2048, eval_tokens=6144,\\n            local_tokens=2048, kd_tokens=2048, holdout_tokens=1024,\\n            epochs=12, patience=3, hidden_eval_tokens=256, latency_repeats=3,\\n        )\\n        return replace(pilot, **kwargs)\\n    if mode not in ("quick", "full"):\\n        raise ValueError("mode must be pilot, quick or full")\\n    cfg = Config() if mode == "quick" else Config(\\n        mode="full",\\n        context_length=256,\\n        eval_stride=128,\\n        eval_batch_size=2,\\n        local_tokens=20480,\\n        kd_tokens=20480,\\n        holdout_tokens=4096,\\n        epochs=15,\\n        eval_tokens=None,\\n        hidden_eval_tokens=8192,\\n        latency_repeats=30,\\n        wiki_train_tokens_per_language=8192,\\n        wiki_validation_tokens_per_language=4096,\\n    )\\n    return replace(cfg, **kwargs)\\n\\n\\ndef validate_config(cfg):\\n    if not 1 <= cfg.eval_stride < cfg.context_length:\\n        raise ValueError("Use overlapping sliding evaluation")\\n    for key in ("local_tokens", "kd_tokens", "holdout_tokens"):\\n        n = getattr(cfg, key)\\n        if n < cfg.context_length or n % cfg.context_length:\\n            raise ValueError(f"{key} must be a positive multiple of context_length")\\n    for key in (\\n        "epochs", "patience", "grad_accumulation", "kl_chunk_tokens",\\n        "latency_seq_len", "latency_repeats", "hidden_eval_tokens",\\n        "cpu_threads", "max_merges", "eval_batch_size",\\n    ):\\n        if getattr(cfg, key) < 1:\\n            raise ValueError(f"{key} must be positive")\\n    if cfg.model_id == "Qwen/Qwen3-0.6B" and cfg.max_merges >= 28:\\n        raise ValueError("Qwen3-0.6B has 28 layers; must leave at least one block")\\n    if cfg.mode == "pilot" and not cfg.fixed_layer_pairs:\\n        raise ValueError("Pilot mode requires fixed pairs")\\n    if cfg.fixed_layer_pairs:\\n        if len(cfg.fixed_layer_pairs) != cfg.max_merges:\\n            raise ValueError("The merge count must match the number of fixed pairs")\\n        seen = set()\\n        for left, right in cfg.fixed_layer_pairs:\\n            if left < 1 or right != left + 1 or left in seen or right in seen:\\n                raise ValueError("Fixed pairs must be adjacent, disjoint, and one-based")\\n            if cfg.model_id == "Qwen/Qwen3-0.6B" and right > 28:\\n                raise ValueError("Fixed pair exceeds the 28 layers of Qwen3-0.6B")\\n            seen.update((left, right))\\n    if cfg.third_pair_candidates:\\n        if cfg.mode != "pilot" or len(cfg.fixed_layer_pairs) != 2:\\n            raise ValueError("Third-pair search requires exactly two fixed pilot pairs")\\n        protected = {4, *(layer for pair in cfg.fixed_layer_pairs for layer in pair)}\\n        if any(4 in pair for pair in cfg.fixed_layer_pairs):\\n            raise ValueError("The two fixed pairs must retain original layer 4")\\n        if len({tuple(pair) for pair in cfg.third_pair_candidates}) != len(cfg.third_pair_candidates):\\n            raise ValueError("Third-pair candidates must be distinct")\\n        for left, right in cfg.third_pair_candidates:\\n            if left < 1 or right != left + 1 or left in protected or right in protected:\\n                raise ValueError("Third-pair candidates must be adjacent and avoid fixed pairs and layer 4")\\n            if cfg.model_id == "Qwen/Qwen3-0.6B" and right > 28:\\n                raise ValueError("Third-pair candidate exceeds the 28 layers of Qwen3-0.6B")\\n    if not math.isfinite(cfg.max_third_ppl_degradation_pct) or cfg.max_third_ppl_degradation_pct < 0:\\n        raise ValueError("The third-merge PPL degradation limit must be finite and nonnegative")\\n    if not 0 <= cfg.math_train_fraction < 1:\\n        raise ValueError("The math training fraction must be at least 0 and below 1")\\n    if cfg.math_train_fraction:\\n        from mathoctopus_training import LANGUAGES\\n\\n        if (not cfg.math_train_languages or len(set(cfg.math_train_languages)) != len(cfg.math_train_languages)\\n                or any(lang not in LANGUAGES.values() for lang in cfg.math_train_languages)):\\n            raise ValueError("Math training languages must be distinct MathOctopus language codes")\\n        if cfg.math_max_window_tokens < 2:\\n            raise ValueError("The math training window limit must be at least two tokens")\\n    if cfg.temperature <= 0 or cfg.epsilon <= 0 or cfg.affine_ridge <= 0:\\n        raise ValueError("temperature, epsilon, affine_ridge must be positive")\\n    if cfg.wiki_train_tokens_per_language < 2 or cfg.wiki_validation_tokens_per_language < 2:\\n        raise ValueError("Wiki-40B train and validation token limits must be positive")\\n    if not cfg.train_languages or len(set(cfg.train_languages)) != len(cfg.train_languages):\\n        raise ValueError("train_languages must contain unique language configs")\\n    if cfg.lambda_hidden < 0 or cfg.lambda_kl < 0 or cfg.lambda_hidden + cfg.lambda_kl == 0:\\n        raise ValueError("Loss weights must be nonnegative and not both zero")\\n    if cfg.lr_transformer <= 0 or cfg.max_grad_norm <= 0 or cfg.weight_decay < 0:\\n        raise ValueError("Invalid optimizer configuration")\\n\\n\\n# ---------------------------------------------------------------------------\\n# Compression-plan utilities\\n# ---------------------------------------------------------------------------\\n\\ndef initial_groups(n_layers):\\n    return tuple((i,) for i in range(n_layers))\\n\\n\\ndef merge_groups(groups, boundary):\\n    """Merge two adjacent *current* groups.\\n\\n    Example (0-based):\\n      ((0,), (1,2), (3,), (4,)) with boundary=1\\n        -> ((0,), (1,2,3), (4,))\\n    """\\n    groups = tuple(tuple(g) for g in groups)\\n    if not 0 <= boundary < len(groups) - 1:\\n        raise ValueError("Invalid current-group boundary")\\n    left, right = groups[boundary], groups[boundary + 1]\\n    if left[-1] + 1 != right[0]:\\n        raise ValueError("Only contiguous original-layer groups may be merged")\\n    combined = left + right\\n    return groups[:boundary] + (combined,) + groups[boundary + 2:]\\n\\n\\ndef validate_groups(groups, n_layers):\\n    flat = [x for group in groups for x in group]\\n    if flat != list(range(n_layers)):\\n        raise ValueError(f"Invalid group partition: {groups}")\\n    if any(group != tuple(range(group[0], group[-1] + 1)) for group in groups):\\n        raise ValueError("Every group must be a contiguous original-layer span")\\n\\n\\ndef fixed_pair_boundary(groups, pair):\\n    """Resolve a one-based original pair after earlier merges shift positions."""\\n    left, right = ((layer - 1,) for layer in pair)\\n    for boundary, (first, second) in enumerate(zip(groups, groups[1:])):\\n        if first == left and second == right:\\n            return boundary\\n    raise ValueError(f"Original layers {pair[0]}-{pair[1]} are not adjacent single layers")\\n\\n\\ndef compressed_groups(groups):\\n    return [tuple(g) for g in groups if len(g) > 1]\\n\\n\\ndef plan_string(groups, one_based=False):\\n    chunks = []\\n    for group in compressed_groups(groups):\\n        vals = [x + 1 if one_based else x for x in group]\\n        chunks.append(str(vals[0]) if len(vals) == 1 else f"{vals[0]}-{vals[-1]}")\\n    return ";".join(chunks) if chunks else "none"\\n\\n\\ndef group_layout(groups, one_based=False):\\n    chunks = []\\n    for group in groups:\\n        vals = [x + 1 if one_based else x for x in group]\\n        chunks.append(str(vals[0]) if len(vals) == 1 else "[" + ",".join(map(str, vals)) + "]")\\n    return " | ".join(chunks)\\n\\n\\ndef transition_kind(groups, boundary):\\n    left, right = groups[boundary], groups[boundary + 1]\\n    if len(left) == 1 and len(right) == 1:\\n        return "new_disjoint_pair"\\n    if len(left) > 1 and len(right) == 1:\\n        return "expand_left_merged_group"\\n    if len(left) == 1 and len(right) > 1:\\n        return "prepend_to_right_merged_group"\\n    return "join_two_merged_groups"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Model/block utilities\\n# ---------------------------------------------------------------------------\\n\\ndef average_blocks_many(blocks):\\n    """Equal parameter average over ALL blocks, not iterative 50/50 averaging."""\\n    if len(blocks) < 2:\\n        raise ValueError("Need at least two blocks to average")\\n    merged = copy.deepcopy(blocks[0])\\n    states = [b.state_dict() for b in blocks]\\n    keys = states[0].keys()\\n    if any(s.keys() != keys for s in states[1:]):\\n        raise ValueError("Blocks have incompatible state_dict keys")\\n    averaged = {}\\n    for key in keys:\\n        tensors = [s[key] for s in states]\\n        if any(t.shape != tensors[0].shape for t in tensors[1:]):\\n            raise ValueError(f"Mismatched tensor for {key}")\\n        first = tensors[0]\\n        if torch.is_floating_point(first) or torch.is_complex(first):\\n            acc = first.clone()\\n            for t in tensors[1:]:\\n                acc = acc + t\\n            averaged[key] = acc / len(tensors)\\n        else:\\n            if any(not torch.equal(first, t) for t in tensors[1:]):\\n                raise ValueError(f"Non-floating buffer differs for {key}")\\n            averaged[key] = first.clone()\\n    merged.load_state_dict(averaged)\\n    return merged\\n\\n\\ndef parameter_digest(model, excluded_prefix=None):\\n    digest = hashlib.sha256()\\n    for name, p in model.named_parameters():\\n        if excluded_prefix and name.startswith(excluded_prefix):\\n            continue\\n        digest.update(name.encode())\\n        digest.update(p.detach().cpu().contiguous().numpy().tobytes())\\n    return digest.hexdigest()\\n\\n\\ndef cpu_state(block):\\n    return {key: value.detach().cpu().clone() for key, value in block.state_dict().items()}\\n\\n\\ndef cleanup():\\n    gc.collect()\\n    if torch.cuda.is_available():\\n        torch.cuda.empty_cache()\\n\\n\\ndef _identity_affine(model):\\n    d = model.config.hidden_size\\n    return np.eye(d, dtype=np.float64), np.zeros(d, dtype=np.float64)\\n\\n\\ndef build_next_student(current, teacher, groups, boundary, method, cfg, affine=None, block_state=None):\\n    """Apply one additional compression step to `current`.\\n\\n    For NonlinearMultiply, the new block is initialized from the CURRENT left\\n    block. Therefore [1,2] -> [1,2,3] starts from the already learned [1,2]\\n    block, then learns the full original teacher span 1..3.\\n    """\\n    if method not in METHODS:\\n        raise ValueError("Unknown method")\\n    validate_groups(groups, len(teacher.model.layers))\\n    new_groups = merge_groups(groups, boundary)\\n    span = new_groups[boundary]\\n    start, end = span[0], span[-1]\\n\\n    student = copy.deepcopy(current).eval().requires_grad_(False)\\n    blocks = list(student.model.layers)\\n    ref = next(student.parameters())\\n\\n    if method == "LinearMultiply":\\n        if affine is None:\\n            affine = _identity_affine(student)\\n        # Keep the replaced (left/current) layer\\\'s mask type rather than\\n        # assuming all Qwen3 variants use global/full attention.\\n        attention_type = getattr(blocks[boundary], "attention_type", "full_attention")\\n        merged = sm.AffineMergedBlock(\\n            *affine, device=ref.device, dtype=ref.dtype,\\n            attention_type=attention_type,\\n        )\\n    elif method == "Remove":\\n        # Keep the first CURRENT block of the combined group.\\n        merged = blocks[boundary]\\n    elif method == "Average":\\n        originals = [teacher.model.layers[i] for i in span]\\n        merged = average_blocks_many(originals).to(ref.device, ref.dtype)\\n    else:  # NonlinearMultiply\\n        # Sequential initialization: preserve the previously merged left group.\\n        merged = copy.deepcopy(blocks[boundary]).to(ref.device)\\n\\n    student.model.layers = nn.ModuleList(blocks[:boundary] + [merged] + blocks[boundary + 2:])\\n    student.config.num_hidden_layers = len(student.model.layers)\\n    student.config.use_cache = student.generation_config.use_cache = False\\n\\n    if block_state is not None:\\n        student.model.layers[boundary].load_state_dict(block_state)\\n\\n    trainable = method == "NonlinearMultiply" and block_state is None\\n    student.requires_grad_(False)\\n    student.model.layers[boundary].requires_grad_(trainable)\\n    student.eval()\\n\\n    expected = {id(p) for p in student.model.layers[boundary].parameters()} if trainable else set()\\n    actual = {id(p) for p in student.parameters() if p.requires_grad}\\n    if actual != expected:\\n        raise AssertionError("Only the new NonlinearMultiply block may be trainable")\\n    if len(student.model.layers) != len(current.model.layers) - 1:\\n        raise AssertionError("Each compression step must remove exactly one current block")\\n    return student, new_groups, (start, end)\\n\\n\\n\\ndef load_wiki40b_balanced(tokenizer, cfg):\\n    """Sample source texts from separate Wiki40B TRAIN and VALIDATION splits.\\n\\n    Streaming bounds memory and avoids downloading the ~38GB full corpus.\\n    Each language contributes its configured number of tokenizer tokens.\\n    """\\n    import datasets\\n    train_blocks, val_blocks, stats = [], [], []\\n    for lang in cfg.train_languages:\\n        row = {"language": lang}\\n        for split, budget, destination in (\\n            ("train", cfg.wiki_train_tokens_per_language, train_blocks),\\n            ("validation", cfg.wiki_validation_tokens_per_language, val_blocks),\\n        ):\\n            stream = datasets.load_dataset("google/wiki40b", lang, split=split,\\n                                           revision=cfg.dataset_revision, streaming=True)\\n            pieces, n, docs = [], 0, 0\\n            for record in stream:\\n                content = record.get("text", "")\\n                if not content or not content.strip():\\n                    continue\\n                # Leave document boundary explicit, no overlap with benchmark sets.\\n                tokens = tokenizer.encode(content + "\\\\n\\\\n", add_special_tokens=False)\\n                if not tokens:\\n                    continue\\n                new = tokens[:budget - n]\\n                pieces.extend(new)\\n                n += len(new)\\n                docs += 1\\n                if n >= budget:\\n                    break\\n            if n < budget:\\n                raise RuntimeError(f"Wiki-40B {lang}/{split}: need {budget} tokens, got {n}")\\n            destination.append(torch.tensor(pieces, dtype=torch.long))\\n            row[split + "_tokens"] = n\\n            row[split + "_documents"] = docs\\n        stats.append(row)\\n        print(f"Wiki-40B {lang}: train={row[\\\'train_tokens\\\']}, validation={row[\\\'validation_tokens\\\']}", flush=True)\\n    train_ids = torch.cat(train_blocks)\\n    val_ids = torch.cat(val_blocks)\\n    if cfg.eval_tokens is not None:\\n        # Preserve language balance even in quick-mode validation.\\n        per_lang = max(2, cfg.eval_tokens // len(val_blocks))\\n        val_ids = torch.cat([block[:per_lang] for block in val_blocks])\\n    return train_ids, val_ids, stats\\n\\n# ---------------------------------------------------------------------------\\n# TRAIN split and objectives\\n# ---------------------------------------------------------------------------\\n\\ndef split_train(ids, cfg):\\n    """Create disjoint LOCAL/KD/HOLDOUT windows from Wiki-40B TRAIN only."""\\n    counts = [cfg.local_tokens, cfg.kd_tokens, cfg.holdout_tokens]\\n    starts = np.arange(0, len(ids) - cfg.context_length + 1, cfg.context_length)\\n    np.random.default_rng(cfg.seed).shuffle(starts)\\n    needed = sum(counts) // cfg.context_length\\n    if len(starts) < needed:\\n        raise ValueError("Not enough disjoint TRAIN windows")\\n    chosen = starts[:needed]\\n    if any(int(s) + cfg.context_length > len(ids) for s in chosen):\\n        raise ValueError("Selected a short final TRAIN window; choose another seed or use more data")\\n    parts, regions, offset = {}, [], 0\\n    for name, count in zip(("local", "kd", "holdout"), counts):\\n        subset = chosen[offset:offset + count // cfg.context_length]\\n        parts[name] = [ids[int(s):int(s) + cfg.context_length].clone() for s in subset]\\n        regions.extend({"Split": name, "Start": int(s), "End": int(s) + cfg.context_length} for s in subset)\\n        offset += len(subset)\\n    if len(set(int(s) for s in chosen)) != needed:\\n        raise AssertionError("TRAIN windows overlap")\\n    return parts, pd.DataFrame(regions)\\n\\n\\n@contextmanager\\ndef capture_teacher_span(model, start, end):\\n    states = {}\\n    handles = [\\n        model.model.layers[start].register_forward_pre_hook(\\n            lambda m, args: states.update(h=args[0])\\n        ),\\n        model.model.layers[end].register_forward_hook(\\n            lambda m, args, out: states.update(target=out)\\n        ),\\n    ]\\n    try:\\n        yield states\\n    finally:\\n        for handle in handles:\\n            handle.remove()\\n\\n\\ndef hidden_loss(student_h, teacher_h, cfg):\\n    if student_h.shape != teacher_h.shape:\\n        raise AssertionError("Teacher/student hidden shape mismatch")\\n    raw = (student_h.float() - teacher_h.float()).square().mean()\\n    return raw / (teacher_h.float().square().mean() + cfg.epsilon), raw\\n\\n\\ndef distillation_kl(student_logits, teacher_logits, cfg):\\n    if student_logits.shape != teacher_logits.shape or student_logits.shape[1] < 2:\\n        raise ValueError("Expected matching [B,T,V] logits, T >= 2")\\n    s = student_logits[:, :-1].reshape(-1, student_logits.shape[-1])\\n    t = teacher_logits[:, :-1].reshape_as(s).detach()\\n\\n    def chunk_loss(a, b):\\n        logp = F.log_softmax(b.float() / cfg.temperature, dim=-1)\\n        logq = F.log_softmax(a.float() / cfg.temperature, dim=-1)\\n        return (logp.exp() * (logp - logq)).sum()\\n\\n    total = s.new_zeros((), dtype=torch.float32)\\n    for pos in range(0, len(s), cfg.kl_chunk_tokens):\\n        a = s[pos:pos + cfg.kl_chunk_tokens]\\n        b = t[pos:pos + cfg.kl_chunk_tokens]\\n        part = checkpoint(chunk_loss, a, b, use_reentrant=False) if a.requires_grad else chunk_loss(a, b)\\n        total = total + part\\n    return cfg.temperature ** 2 * total / len(s)\\n\\n\\n@contextmanager\\ndef capture_student_layer(model, boundary):\\n    states = {}\\n    handle = model.model.layers[boundary].register_forward_hook(\\n        lambda m, args, output: states.update(output=output)\\n    )\\n    try:\\n        yield states\\n    finally:\\n        handle.remove()\\n\\n\\ndef local_objective_span(teacher, student, boundary, span, ids, cfg):\\n    # Full-model forward supplies RoPE position_embeddings and causal mask,\\n    # unlike a bare direct call to Qwen3DecoderLayer.\\n    start, end = span\\n    with torch.no_grad(), capture_teacher_span(teacher, start, end) as target:\\n        teacher.model(ids, use_cache=False, return_dict=True)\\n    with capture_student_layer(student, boundary) as states:\\n        student.model(ids, use_cache=False, return_dict=True)\\n    return hidden_loss(states["output"], target["target"], cfg)\\n\\n\\ndef math_response_loss(logits, ids, response_start):\\n    """Use the logit immediately before each assistant token, including EOS."""\\n    return F.cross_entropy(logits[0, response_start - 1:-1].float(), ids[0, response_start:])\\n\\n\\ndef kd_objectives(teacher, student, ids, cfg, response_start):\\n    with torch.no_grad():\\n        teacher_logits = teacher(ids, use_cache=False, return_dict=True).logits\\n    student_logits = student(ids, use_cache=False, return_dict=True).logits\\n    kl = distillation_kl(student_logits, teacher_logits, cfg)\\n    response = None if response_start is None else math_response_loss(student_logits, ids, response_start)\\n    return kl, response\\n\\n\\n@torch.no_grad()\\ndef reconstruction_metrics_span(teacher, student, boundary, span, windows, cfg, response_starts=None):\\n    device = next(teacher.parameters()).device\\n    sqerr = energy = kl_sum = response_sum = 0.0\\n    elements = tokens = response_windows = 0\\n    normalized = []\\n    start, end = span\\n    for index, window in enumerate(windows):\\n        ids = window[None].to(device)\\n        with capture_teacher_span(teacher, start, end) as states:\\n            teacher_logits = teacher(ids, use_cache=False, return_dict=True).logits\\n        with capture_student_layer(student, boundary) as student_states:\\n            student_logits = student(ids, use_cache=False, return_dict=True).logits\\n        sh = student_states["output"]\\n        hn, raw = hidden_loss(sh, states["target"], cfg)\\n        sqerr += float(raw) * sh.numel()\\n        energy += float(states["target"].float().square().sum())\\n        elements += sh.numel()\\n        normalized.append(float(hn))\\n        count = ids.shape[1] - 1\\n        kl_sum += float(distillation_kl(student_logits, teacher_logits, cfg)) * count\\n        tokens += count\\n        if response_starts is not None and response_starts[index] is not None:\\n            response_sum += float(math_response_loss(student_logits, ids, response_starts[index]))\\n            response_windows += 1\\n    raw = sqerr / elements\\n    kl = kl_sum / tokens\\n    return {\\n        "Hidden_MSE": raw,\\n        "Normalized_Hidden_MSE": raw / (energy / elements + cfg.epsilon),\\n        "Mean_Window_Normalized_Hidden_MSE": float(np.mean(normalized)),\\n        "Teacher_KL": kl,\\n        "Teacher_KL_Unscaled": kl / cfg.temperature ** 2,\\n        "Diagnostic_Tokens": sum(len(w) for w in windows),\\n        "KL_Predicted_Tokens": tokens,\\n        "Math_Response_CE": response_sum / response_windows if response_windows else None,\\n        "Math_Response_CE_Per_Window": response_sum / len(windows),\\n    }\\n\\n\\ndef train_student_span(teacher, student, boundary, span, parts, cfg, stage, history_path=None):\\n    block = student.model.layers[boundary]\\n    frozen_before = parameter_digest(student, f"model.layers.{boundary}.")\\n    optimizer = torch.optim.AdamW(\\n        block.parameters(), lr=cfg.lr_transformer, weight_decay=cfg.weight_decay\\n    )\\n    device = next(student.parameters()).device\\n    history, best_loss, best_epoch, best_state, stale = [], math.inf, 0, None, 0\\n    steps = max(len(parts["local"]), len(parts["kd"]))\\n    start, end = span\\n    response_starts = parts.get("response_starts", {})\\n    kd_starts = response_starts.get("kd", [None] * len(parts["kd"]))\\n\\n    def holdout_row(epoch, train=None):\\n        metric = reconstruction_metrics_span(\\n            teacher, student, boundary, span, parts["holdout"], cfg, response_starts.get("holdout")\\n        )\\n        h = metric["Mean_Window_Normalized_Hidden_MSE"]\\n        k = metric["Teacher_KL"]\\n        math_ce = metric["Math_Response_CE_Per_Window"]\\n        row = {\\n            "Stage": stage,\\n            "Span": f"{start}-{end}",\\n            "Boundary": boundary,\\n            "Method": "NonlinearMultiply",\\n            "Epoch": epoch,\\n            "Holdout_Hidden_Loss": h,\\n            "Holdout_KL": k,\\n            "Holdout_Math_Response_CE": metric["Math_Response_CE"],\\n            "Holdout_Total_Loss": cfg.lambda_hidden * h + cfg.lambda_kl * k + MATH_RESPONSE_WEIGHT * math_ce,\\n        }\\n        row.update(train or {\\n            "Train_Hidden_Loss": None,\\n            "Train_KL": None,\\n            "Train_Math_Response_CE": None,\\n            "Train_Total_Loss": None,\\n        })\\n        if not math.isfinite(row["Holdout_Total_Loss"]):\\n            raise ArithmeticError("Non-finite HOLDOUT loss")\\n        return row\\n\\n    for epoch in range(cfg.epochs + 1):\\n        train = None\\n        if epoch:\\n            rng = np.random.default_rng(\\n                cfg.seed + stage * 100003 + start * 1009 + end * 97 + epoch\\n            )\\n            order_local = rng.permutation(len(parts["local"]))\\n            order_kd = rng.permutation(len(parts["kd"]))\\n            hsum = ksum = response_sum = 0.0\\n            response_windows = 0\\n            optimizer.zero_grad(set_to_none=True)\\n            for step in range(steps):\\n                group_start = (step // cfg.grad_accumulation) * cfg.grad_accumulation\\n                divisor = min(cfg.grad_accumulation, steps - group_start)\\n                local = parts["local"][order_local[step % len(order_local)]][None].to(device)\\n                kd_index = order_kd[step % len(order_kd)]\\n                kd = parts["kd"][kd_index][None].to(device)\\n\\n                hn, raw = local_objective_span(teacher, student, boundary, span, local, cfg)\\n                if not torch.isfinite(hn):\\n                    raise ArithmeticError("Non-finite hidden loss")\\n                hsum += float(hn.detach())\\n                if cfg.lambda_hidden:\\n                    (cfg.lambda_hidden * hn / divisor).backward()\\n                del hn, raw\\n\\n                kl, response = kd_objectives(teacher, student, kd, cfg, kd_starts[kd_index])\\n                if not torch.isfinite(kl):\\n                    raise ArithmeticError("Non-finite KL")\\n                ksum += float(kl.detach())\\n                kd_loss = cfg.lambda_kl * kl\\n                if response is not None:\\n                    if not torch.isfinite(response):\\n                        raise ArithmeticError("Non-finite math response loss")\\n                    response_sum += float(response.detach())\\n                    response_windows += 1\\n                    kd_loss = kd_loss + MATH_RESPONSE_WEIGHT * response\\n                if cfg.lambda_kl or response is not None:\\n                    (kd_loss / divisor).backward()\\n                del kl, response, kd_loss\\n\\n                if (step + 1) % cfg.grad_accumulation == 0 or step + 1 == steps:\\n                    torch.nn.utils.clip_grad_norm_(\\n                        block.parameters(), cfg.max_grad_norm, error_if_nonfinite=True\\n                    )\\n                    optimizer.step()\\n                    optimizer.zero_grad(set_to_none=True)\\n\\n            h, k = hsum / steps, ksum / steps\\n            train = {\\n                "Train_Hidden_Loss": h,\\n                "Train_KL": k,\\n                "Train_Math_Response_CE": response_sum / response_windows if response_windows else None,\\n                "Train_Total_Loss": (cfg.lambda_hidden * h + cfg.lambda_kl * k\\n                                     + MATH_RESPONSE_WEIGHT * response_sum / steps),\\n            }\\n\\n        row = holdout_row(epoch, train)\\n        history.append(row)\\n        if history_path:\\n            pd.DataFrame(history).to_csv(history_path, index=False)\\n        score = row["Holdout_Total_Loss"]\\n        if score < best_loss:\\n            best_loss, best_epoch, best_state, stale = score, epoch, cpu_state(block), 0\\n        else:\\n            stale += 1\\n        print(\\n            f"    Nonlinear span={start}-{end} epoch={epoch} "\\n            f"holdout={score:.6f} best={best_epoch}",\\n            flush=True,\\n        )\\n        if epoch and stale >= cfg.patience:\\n            break\\n\\n    block.load_state_dict(best_state)\\n    if parameter_digest(student, f"model.layers.{boundary}.") != frozen_before:\\n        raise AssertionError("Frozen weights changed")\\n    return history, best_epoch\\n\\n\\n# ---------------------------------------------------------------------------\\n# Affine multi-layer composition\\n# ---------------------------------------------------------------------------\\n\\n@torch.inference_mode()\\ndef collect_span_states(model, windows, start, end):\\n    """Return [input_to_start, output_start, ..., output_end] as CPU arrays."""\\n    d = model.config.hidden_size\\n    buckets = [[] for _ in range(end - start + 2)]\\n\\n    def append(slot, tensor):\\n        buckets[slot].append(tensor.detach().reshape(-1, d).float().cpu().numpy())\\n\\n    handles = [\\n        model.model.layers[start].register_forward_pre_hook(\\n            lambda m, args: append(0, args[0])\\n        )\\n    ]\\n    for layer in range(start, end + 1):\\n        slot = layer - start + 1\\n        handles.append(\\n            model.model.layers[layer].register_forward_hook(\\n                lambda m, args, out, slot=slot: append(slot, out)\\n            )\\n        )\\n    try:\\n        device = next(model.parameters()).device\\n        for window in windows:\\n            model.model(window[None].to(device), use_cache=False, return_dict=True)\\n    finally:\\n        for handle in handles:\\n            handle.remove()\\n    return [np.concatenate(parts, axis=0) for parts in buckets]\\n\\n\\ndef fit_affine_span(teacher, windows, span, cfg):\\n    start, end = span\\n    states = collect_span_states(teacher, windows, start, end)\\n    d = teacher.config.hidden_size\\n    A_total = np.eye(d, dtype=np.float64)\\n    b_total = np.zeros(d, dtype=np.float64)\\n    with threadpool_limits(limits=cfg.cpu_threads):\\n        for x, y in zip(states[:-1], states[1:]):\\n            A, b = sm.fit_affine(x, y, cfg.affine_ridge)\\n            b_total = b_total @ A + b\\n            A_total = A_total @ A\\n    return A_total, b_total\\n\\n\\n# ---------------------------------------------------------------------------\\n# Checkpoint/replay: save only the NEW merged block at each stage\\n# ---------------------------------------------------------------------------\\n\\ndef save_stage_delta(path, teacher, method, stage, groups_before, groups_after,\\n                     boundary, span, student, cfg, best_epoch):\\n    payload = {\\n        "format_version": 2,\\n        "method": method,\\n        "stage": stage,\\n        "boundary": boundary,\\n        "span": tuple(span),\\n        "groups_before": tuple(tuple(g) for g in groups_before),\\n        "groups_after": tuple(tuple(g) for g in groups_after),\\n        "model_id": cfg.model_id,\\n        "model_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,\\n        "original_layers": len(teacher.model.layers),\\n        "config": asdict(cfg),\\n        "best_epoch": best_epoch,\\n        "block_state": None,\\n    }\\n    if method in ("LinearMultiply", "NonlinearMultiply"):\\n        payload["block_state"] = cpu_state(student.model.layers[boundary])\\n    torch.save(payload, path)\\n\\n\\ndef replay_selected_stage(run_dir, method, stage, teacher=None, device=None):\\n    out = Path(run_dir)\\n    cfg = Config(**json.loads((out / "configuration.json").read_text(encoding="utf-8")))\\n    if teacher is None:\\n        first = torch.load(out / f"selected_{method}_stage1.pt", map_location="cpu", weights_only=True)\\n        teacher = sm.AutoModelForCausalLM.from_pretrained(\\n            first["model_id"], revision=first["model_revision"],\\n            attn_implementation="eager", dtype=torch.float32,\\n        ).to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval().requires_grad_(False)\\n    current = copy.deepcopy(teacher).eval().requires_grad_(False)\\n    current.config.use_cache = current.generation_config.use_cache = False\\n    groups = initial_groups(len(teacher.model.layers))\\n\\n    for s in range(1, stage + 1):\\n        saved = torch.load(out / f"selected_{method}_stage{s}.pt", map_location="cpu", weights_only=True)\\n        expected_before = tuple(tuple(g) for g in saved["groups_before"])\\n        if tuple(tuple(g) for g in groups) != expected_before:\\n            raise ValueError(f"Stage replay mismatch for {method}, stage {s}")\\n        boundary = int(saved["boundary"])\\n        if method == "LinearMultiply":\\n            affine = _identity_affine(current)\\n        else:\\n            affine = None\\n        current, new_groups, span = build_next_student(\\n            current, teacher, groups, boundary, method, cfg,\\n            affine=affine, block_state=saved.get("block_state"),\\n        )\\n        current.requires_grad_(False).eval()\\n        groups = new_groups\\n        if tuple(span) != tuple(saved["span"]):\\n            raise ValueError("Saved span does not match replayed span")\\n    return current, groups\\n\\n\\n# ---------------------------------------------------------------------------\\n# Evaluation and greedy search\\n# ---------------------------------------------------------------------------\\n\\ndef evaluate_ppl(model, ids, cfg):\\n    return ep.calculate_ppl(\\n        model,\\n        ids,\\n        cfg.context_length,\\n        cfg.eval_stride,\\n        protocol="sliding",\\n        batch_size=cfg.eval_batch_size,\\n    )\\n\\n\\ndef _candidate_row(method, stage, groups_before, groups_after, boundary, span,\\n                   metric, baseline, params, best_epoch, transition):\\n    start, end = span\\n    return {\\n        "Method": method,\\n        "Compression_Steps": stage,\\n        "Candidate_Boundary": boundary,\\n        "Transition": transition,\\n        "Merged_Span_0based": f"{start}-{end}",\\n        "Merged_Span_1based": f"{start+1}-{end+1}",\\n        "Plan_0based": plan_string(groups_after, one_based=False),\\n        "Plan_1based": plan_string(groups_after, one_based=True),\\n        "Layout_0based": group_layout(groups_after, one_based=False),\\n        "Layout_1based": group_layout(groups_after, one_based=True),\\n        "Validation_PPL": metric["PPL"],\\n        "Validation_NLL": metric["NLL"],\\n        "Predicted_Tokens": metric["Predicted_Tokens"],\\n        "Original_Validation_PPL": baseline["PPL"],\\n        "PPL_Delta_vs_Original": metric["PPL"] - baseline["PPL"],\\n        "PPL_Degradation_%": 100 * (metric["PPL"] / baseline["PPL"] - 1),\\n        "Parameters": params,\\n        "Parameter_Reduction_%": 100 * (1 - params / baseline["Parameters"]),\\n        "Remaining_Blocks": len(groups_after),\\n        "Best_Epoch": best_epoch,\\n        "Groups_Before": repr(tuple(tuple(g) for g in groups_before)),\\n        "Groups_After": repr(tuple(tuple(g) for g in groups_after)),\\n    }\\n\\n\\ndef greedy_method_search(teacher, method, parts, val_ids, diag_windows, cfg, out, baseline):\\n    """Greedy stage-wise search for one method.\\n\\n    Fixed pilot stages use their prescribed original-layer pairs. An optional\\n    third stage compares a shortlist from the same frozen stage-2 checkpoint.\\n    """\\n    fixed_pairs = cfg.fixed_layer_pairs\\n    current = teacher if fixed_pairs else copy.deepcopy(teacher).eval().requires_grad_(False)\\n    current.config.use_cache = current.generation_config.use_cache = False\\n    groups = initial_groups(len(teacher.model.layers))\\n    all_rows, selected_rows, histories = [], [], []\\n\\n    planned_stages = cfg.max_merges + bool(cfg.third_pair_candidates)\\n    for stage in range(1, planned_stages + 1):\\n        optional_stage = stage > cfg.max_merges\\n        fixed_stage = bool(fixed_pairs) and not optional_stage\\n        print(f"\\\\n===== {method}: compression stage {stage}/{planned_stages} =====", flush=True)\\n        best = None\\n        best_block_state = None\\n        best_affine = None\\n        best_groups = None\\n        best_span = None\\n        best_boundary = None\\n\\n        if optional_stage:\\n            boundaries = tuple(fixed_pair_boundary(groups, pair) for pair in cfg.third_pair_candidates)\\n        elif fixed_stage:\\n            pair = fixed_pairs[stage - 1]\\n            boundaries = (fixed_pair_boundary(groups, pair),)\\n            print(f"  fixed original layers (1-based): {pair[0]}-{pair[1]}", flush=True)\\n        else:\\n            boundaries = range(len(groups) - 1)\\n        for boundary in boundaries:\\n            candidate_groups = merge_groups(groups, boundary)\\n            span_tuple = candidate_groups[boundary]\\n            span = (span_tuple[0], span_tuple[-1])\\n            transition = transition_kind(groups, boundary)\\n            print(\\n                f"  candidate boundary={boundary} span={span[0]}-{span[1]} "\\n                f"type={transition}", flush=True\\n            )\\n\\n            affine = fit_affine_span(teacher, parts["local"], span, cfg) if method == "LinearMultiply" else None\\n            sm.set_seed(cfg.seed + stage * 100003 + boundary * 1009 + span[0] * 97)\\n            student, new_groups, built_span = build_next_student(\\n                current, teacher, groups, boundary, method, cfg, affine=affine\\n            )\\n            if built_span != span:\\n                raise AssertionError("Span bookkeeping mismatch")\\n            if fixed_stage:\\n                # There is one candidate. Release the prior student before training.\\n                current = student\\n\\n            best_epoch = None\\n            if method == "NonlinearMultiply":\\n                hist_path = out / f"history_{method}_stage{stage}_b{boundary}_span{span[0]}_{span[1]}.csv"\\n                hist, best_epoch = train_student_span(\\n                    teacher, student, boundary, span, parts, cfg, stage, hist_path\\n                )\\n                histories.extend(hist)\\n                pd.DataFrame(histories).to_csv(out / f"training_history_{method}.csv", index=False)\\n\\n            metric = evaluate_ppl(student, val_ids, cfg)\\n            params = sm.parameter_count(student)\\n            row = _candidate_row(\\n                method, stage, groups, new_groups, boundary, span,\\n                metric, baseline, params, best_epoch, transition,\\n            )\\n            # Use validation NLL for greedy selection and report it for fixed pairs.\\n            # Run hidden/KL diagnostics after the stage.\\n            row["Candidate_Pair_1based"] = f"{span[0] + 1}-{span[1] + 1}"\\n            row["Optional_Stage"] = optional_stage\\n            row["Within_Optional_PPL_Budget"] = (\\n                row["PPL_Degradation_%"] <= cfg.max_third_ppl_degradation_pct if optional_stage else None\\n            )\\n            all_rows.append(row)\\n            pd.DataFrame(all_rows).to_csv(out / f"candidates_{method}.csv", index=False)\\n\\n            if best is None or metric["NLL"] < best["Validation_NLL"]:\\n                best = dict(row)\\n                best_block_state = None if fixed_stage else cpu_state(student.model.layers[boundary])\\n                best_affine = affine\\n                best_groups = new_groups\\n                best_span = span\\n                best_boundary = boundary\\n\\n            print(\\n                f"    PPL={metric[\\\'PPL\\\']:.4f}, degradation={row[\\\'PPL_Degradation_%\\\']:+.2f}%",\\n                flush=True,\\n            )\\n            del student\\n            cleanup()\\n\\n        if best is None:\\n            raise RuntimeError("No candidate was evaluated")\\n\\n        if optional_stage and not best["Within_Optional_PPL_Budget"]:\\n            print(f"  >>> Third merge rejected. Best candidate {best[\\\'Candidate_Pair_1based\\\']} "\\n                  f"degraded validation PPL by {best[\\\'PPL_Degradation_%\\\']:.2f}%, "\\n                  f"above the {cfg.max_third_ppl_degradation_pct:.2f}% limit. Keeping stage 2.", flush=True)\\n            break\\n\\n        # Replay a shortlisted winner. A fixed stage already owns its sole candidate.\\n        if fixed_stage:\\n            check_groups = new_groups\\n        elif method == "LinearMultiply":\\n            current, check_groups, _ = build_next_student(\\n                current, teacher, groups, best_boundary, method, cfg,\\n                affine=best_affine, block_state=best_block_state,\\n            )\\n        else:\\n            current, check_groups, _ = build_next_student(\\n                current, teacher, groups, best_boundary, method, cfg,\\n                block_state=best_block_state,\\n            )\\n        current.requires_grad_(False).eval()\\n        if check_groups != best_groups:\\n            raise AssertionError("Selected-plan reconstruction mismatch")\\n\\n        if diag_windows:\\n            best.update(reconstruction_metrics_span(\\n                teacher, current, best_boundary, best_span, diag_windows, cfg\\n            ))\\n\\n        selected_rows.append(best)\\n        pd.DataFrame(selected_rows).to_csv(out / f"selected_stages_{method}.csv", index=False)\\n        save_stage_delta(\\n            out / f"selected_{method}_stage{stage}.pt",\\n            teacher, method, stage, groups, best_groups, best_boundary,\\n            best_span, current, cfg, best["Best_Epoch"],\\n        )\\n        print(\\n            f"  >>> SELECTED {method} stage={stage}: "\\n            f"original-layer plan={best[\\\'Plan_1based\\\']} PPL={best[\\\'Validation_PPL\\\']:.4f}",\\n            flush=True,\\n        )\\n        groups = best_groups\\n\\n    return all_rows, selected_rows\\n\\n\\ndef run_experiment(cfg=None):\\n    cfg = cfg or make_config()\\n    validate_config(cfg)\\n    sm.check_transformers_version()\\n    sm.set_seed(cfg.seed)\\n    torch.set_num_threads(cfg.cpu_threads)\\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\\n\\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\\n    out = Path(cfg.output_dir) / f"multi_merge_{cfg.mode}_{stamp}"\\n    out.mkdir(parents=True, exist_ok=False)\\n    ep._write_json(out / "configuration.json", asdict(cfg))\\n    fixed = bool(cfg.fixed_layer_pairs)\\n    methods = ("NonlinearMultiply",) if fixed else METHODS\\n    selection_basis = (\\n        "Fixed original layer pairs in the stated order; validation NLL is reported."\\n        if fixed else\\n        "At each stage, evaluate every adjacent CURRENT-group merge and select minimum Wiki-40B validation NLL."\\n    )\\n    if cfg.third_pair_candidates:\\n        selection_basis = (\\n            "Train the two fixed pairs, then rank third-pair candidates by Wiki-40B validation NLL. "\\n            f"Accept the best third pair only within +{cfg.max_third_ppl_degradation_pct:g}% "\\n            "validation perplexity of Original. Otherwise retain stage 2."\\n        )\\n    ep._write_json(out / "protocol.json", {\\n        "name": "Multi-Step Adjacent-Layer Compression of Qwen3",\\n        "methods": methods,\\n        "max_merges": cfg.max_merges,\\n        "fixed_layer_pairs_1based": cfg.fixed_layer_pairs,\\n        "third_pair_candidates_1based": cfg.third_pair_candidates,\\n        "max_third_ppl_degradation_pct": cfg.max_third_ppl_degradation_pct,\\n        "selection": selection_basis,\\n        "overlap_semantics": "A selected merged group remains one current block. Merging it with its neighbor expands the original span, e.g. [1,2]+[3] -> [1,2,3].",\\n        "nonlinear_target": "New nonlinear block is initialized from current left block, but hidden target is the ORIGINAL teacher mapping across the full combined span.",\\n        "average": "Equal parameter average over all ORIGINAL blocks in the combined span.",\\n        "remove": "Keep first block of combined group; remove the rest.",\\n        "test_policy": "MMMLU, INCLUDE, and MGSM test examples are excluded from training and validation selection.",\\n        "math_training_fraction_by_window": cfg.math_train_fraction,\\n        "math_response_loss": {\\n            "weight": MATH_RESPONSE_WEIGHT if cfg.math_train_fraction else 0.0,\\n            "target": "Assistant solution, canonical GSM8K TRAIN final answer, and EOS",\\n            "applied_to": "MathOctopus KD windows and the corresponding training holdout",\\n        },\\n        "search_limitation": (\\n            "Only the prescribed pairs and optional shortlist are evaluated." if fixed else\\n            "Greedy stage-wise search does not guarantee a globally optimal plan."\\n        ),\\n    })\\n\\n    import datasets\\n    tokenizer = sm.AutoTokenizer.from_pretrained(cfg.model_id, revision=cfg.model_revision)\\n    tokenizer.model_max_length = 10**12\\n    # Stream multilingual Wiki-40B; the old WikiText-2 source is completely removed.\\n    train_ids, val_ids, dataset_stats = load_wiki40b_balanced(tokenizer, cfg)\\n    training_datasets = ["google/wiki40b"]\\n    if cfg.math_train_fraction:\\n        from mathoctopus_training import MATH_DATASET, split_mixed_train\\n\\n        parts, regions, provenance = split_mixed_train(tokenizer, train_ids, cfg, split_train)\\n        training_datasets.append(MATH_DATASET)\\n        ep._write_json(out / "mathoctopus_data_stats.json", provenance)\\n    else:\\n        parts, regions = split_train(train_ids, cfg)\\n        regions = regions.assign(Dataset="google/wiki40b", Tokens=regions["End"] - regions["Start"])\\n    regions.to_csv(out / "train_regions.csv", index=False)\\n    source_mix = regions.groupby(["Split", "Dataset"], as_index=False).agg(\\n        Windows=("Tokens", "size"), Tokens=("Tokens", "sum"),\\n    )\\n    source_mix.to_csv(out / "train_source_mix.csv", index=False)\\n    print("Training and holdout windows by source:\\\\n" + source_mix.to_string(index=False), flush=True)\\n    ep._write_json(out / "wiki40b_data_stats.json", dataset_stats)\\n\\n    teacher = sm.AutoModelForCausalLM.from_pretrained(\\n        cfg.model_id, revision=cfg.model_revision,\\n        attn_implementation="eager", dtype=torch.float32,\\n    ).to(device).eval().requires_grad_(False)\\n    teacher.config.use_cache = teacher.generation_config.use_cache = False\\n    layers = len(teacher.model.layers)\\n    if cfg.max_merges >= layers:\\n        raise ValueError("max_merges must be smaller than the number of transformer blocks")\\n    if any(right > layers for _, right in (*cfg.fixed_layer_pairs, *cfg.third_pair_candidates)):\\n        raise ValueError(f"A fixed pair exceeds the {layers} layers of {cfg.model_id}")\\n    if cfg.model_id == "Qwen/Qwen3-0.6B" and layers != 28:\\n        raise AssertionError("Expected 28 Qwen3-0.6B blocks")\\n    math_context = cfg.math_max_window_tokens if cfg.math_train_fraction else 0\\n    if max(cfg.context_length, cfg.latency_seq_len, math_context) > teacher.config.max_position_embeddings:\\n        raise ValueError("Sequence length exceeds Qwen3 context")\\n\\n    teacher_digest = parameter_digest(teacher)\\n    ep._write_json(out / "environment.json", {\\n        "python": platform.python_version(),\\n        "torch": str(torch.__version__),\\n        "transformers": sm.transformers.__version__,\\n        "numpy": np.__version__,\\n        "pandas": pd.__version__,\\n        "scipy": sm.scipy.__version__,\\n        "datasets": datasets.__version__,\\n        "device": str(device),\\n        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,\\n        "model_resolved_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,\\n        "dataset_requested_revision": cfg.dataset_revision,\\n        "dataset_languages": list(cfg.train_languages),\\n        "dataset_source": "google/wiki40b",\\n        "training_datasets": training_datasets,\\n        "validation_dataset": "google/wiki40b",\\n        "teacher_parameter_sha256": teacher_digest,\\n    })\\n\\n    baseline = {\\n        **evaluate_ppl(teacher, val_ids, cfg),\\n        "Parameters": sm.parameter_count(teacher),\\n    }\\n    ep._write_json(out / "baseline_validation.json", baseline)\\n    diag_windows = sm.make_state_windows(\\n        val_ids, min(cfg.hidden_eval_tokens, len(val_ids)), cfg.context_length\\n    )\\n    diag_windows = [w for w in diag_windows if len(w) >= 2]\\n\\n    print(f"Device: {device}; output: {out}", flush=True)\\n    print(f"Original validation: {baseline}", flush=True)\\n\\n    candidate_rows, selected_rows = [], []\\n    for method in methods:\\n        rows, selected = greedy_method_search(\\n            teacher, method, parts, val_ids, diag_windows, cfg, out, baseline\\n        )\\n        candidate_rows.extend(rows)\\n        selected_rows.extend(selected)\\n        pd.DataFrame(candidate_rows).to_csv(out / "all_candidates.csv", index=False)\\n        pd.DataFrame(selected_rows).to_csv(out / "selected_by_stage.csv", index=False)\\n\\n    if parameter_digest(teacher) != teacher_digest:\\n        raise AssertionError("Original teacher changed")\\n\\n    selected_frame = pd.DataFrame(selected_rows)\\n    selected_frame = selected_frame.sort_values(["Compression_Steps", "Method"]).reset_index(drop=True)\\n    selected_frame.to_csv(out / "selected_by_stage.csv", index=False)\\n\\n    pivot = selected_frame.pivot(\\n        index="Compression_Steps", columns="Method", values="Validation_PPL"\\n    ).reindex(columns=methods)\\n    pivot.to_csv(out / "validation_ppl_by_compression.csv")\\n\\n    final_stage = int(selected_frame["Compression_Steps"].max())\\n    checkpoint_names = [f"selected_{row.Method}_stage{row.Compression_Steps}.pt"\\n                        for row in selected_frame.itertuples(index=False)]\\n    lock_files = ["configuration.json", "selected_by_stage.csv", *checkpoint_names]\\n    ep._write_json(out / "selection_locked.json", {\\n        "basis": selection_basis + " Nonlinear epoch selected on TRAIN HOLDOUT only.",\\n        "max_merges": final_stage,\\n        "file_sha256": {\\n            name: hashlib.sha256((out / name).read_bytes()).hexdigest()\\n            for name in lock_files\\n        },\\n    })\\n\\n    del teacher\\n    cleanup()\\n    return {\\n        "output_dir": str(out),\\n        "methods": methods,\\n        "final_stage": final_stage,\\n        "candidates": pd.DataFrame(candidate_rows),\\n        "selected": selected_frame,\\n        "pivot": pivot,\\n        "baseline": baseline,\\n    }\\n\\n\\n\', \'qwen3_benchmarks.py\': \'\\n"""Benchmark-only few-shot evaluation of original and compressed Qwen3.\\n\\nDefault shots are 5 for MMMLU and INCLUDE and 8 for MGSM. A pilot can use fewer.\\nDemonstrations use MMLU DEV (same subject), INCLUDE VALIDATION, or MGSM TRAIN.\\nScored questions are drawn from each benchmark\\\'s TEST split.\\nAll prompts are generated once, saved, and reused identically for baseline and\\ncompressed models. 100k tokens is a PER BENCHMARK/PER LANGUAGE INPUT-PROMPT cap;\\nquestion sets are finite, so the actual scored tokens can be less.\\n"""\\nfrom __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport random\\nimport re\\nimport unicodedata\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch.nn import functional as F\\n\\nimport qwen3_compression_multi as nc\\nfrom strict_mgsm import classify\\n\\nLANGUAGES = {\\n    "de": {"wiki": "de", "mmmlu": "DE_DE", "include": "German", "mgsm": "de"},\\n    "es": {"wiki": "es", "mmmlu": "ES_LA", "include": "Spanish", "mgsm": "es"},\\n    "fr": {"wiki": "fr", "mmmlu": "FR_FR", "include": "French", "mgsm": "fr"},\\n    "ja": {"wiki": "ja", "mmmlu": "JA_JP", "include": "Japanese", "mgsm": "ja"},\\n    "ru": {"wiki": "ru", "mmmlu": "RU_RU", "include": "Russian", "mgsm": "ru"},\\n    "zh": {"wiki": "zh-cn", "mmmlu": "ZH_CN", "include": "Chinese", "mgsm": "zh"},\\n}\\n# MMMLU has no Russian locale. Restrict to the five common languages across\\n# *all three* benchmarks, or benchmark-specific subsets can be requested.\\nMMMLU_LOCALES = set("AR_XY BN_BD DE_DE ES_LA FR_FR HI_IN ID_ID IT_IT JA_JP KO_KR PT_BR SW_KE YO_NG ZH_CN".split())\\nBENCHMARKS = ("MMMLU", "INCLUDE", "MGSM")\\nMCQ_LETTERS = "ABCD"\\nMCQ_ASSISTANT_PREFILL = "Answer:"\\nOLD_MGSM_INSTRUCTION = "Solve the following math problems. Show your reasoning and end with \\\'Final answer: <number>\\\'."\\nMGSM_INSTRUCTION = (\\n    "Solve the following math problem. Keep your reasoning brief. End with a separate line "\\n    "\\\'Final answer: <number>\\\' containing just one number, with no units or thousands separators. "\\n    "Stop after that line."\\n)\\n\\n\\ndef normalized_lang(lang):\\n    if lang not in LANGUAGES:\\n        raise ValueError(f"Unsupported language key {lang!r}; available {sorted(LANGUAGES)}")\\n    return LANGUAGES[lang]\\n\\n\\ndef normalize_answer(row, kind):\\n    if kind == "MMMLU":\\n        a = str(row["Answer"]).strip().upper()\\n        if a not in MCQ_LETTERS:\\n            raise ValueError(f"Unknown MMMLU answer: {a}")\\n        return a\\n    x = row["answer"]\\n    a = MCQ_LETTERS[int(x)]\\n    return a\\n\\n\\ndef normalize_mcq(row, kind):\\n    if kind == "MMMLU":\\n        opts = [str(row[k]) for k in MCQ_LETTERS]\\n        return str(row["Question"]), opts, normalize_answer(row, kind)\\n    opts = row.get("choices")\\n    if not isinstance(opts, list):\\n        opts = [row[f"option_{l.lower()}"] for l in MCQ_LETTERS]\\n    if len(opts) != 4:\\n        raise ValueError("INCLUDE expected four choices")\\n    return str(row["question"]), list(map(str, opts)), normalize_answer(row, kind)\\n\\n\\ndef mcq_example(row, kind, is_demo):\\n    question, options, answer = normalize_mcq(row, kind)\\n    block = f"Question: {question}\\\\n" + "\\\\n".join(\\n        f"{letter}. {option}" for letter, option in zip(MCQ_LETTERS, options)\\n    ) + "\\\\nAnswer:"\\n    if is_demo:\\n        return block + " " + answer + "\\\\n\\\\n"\\n    return block\\n\\n\\ndef mgsm_example(row, is_demo):\\n    question = str(row["question"]).removeprefix("Question: ")\\n    if is_demo:\\n        # Explanations are taken ONLY from MGSM train, not TEST.\\n        rationale = str(row.get("answer", "")).strip()\\n        number = str(row["answer_number"])\\n        return f"Question: {question}\\\\nSolution: {rationale}\\\\nFinal answer: {number}\\\\n\\\\n"\\n    return f"Question: {question}\\\\nSolution:"\\n\\n\\ndef to_chat_prompt(tokenizer, content):\\n    return tokenizer.apply_chat_template(\\n        [{"role": "user", "content": content}],\\n        tokenize=False, add_generation_prompt=True, enable_thinking=False,\\n    )\\n\\n\\ndef mcq_continuations(tokenizer, scoring_prompt, prefix_ids):\\n    """Tokenize the actual continuation after the assistant-side answer cue."""\\n    continuations = []\\n    prefix_length = len(prefix_ids)\\n    for letter in MCQ_LETTERS:\\n        full = tokenizer.encode(scoring_prompt + " " + letter, add_special_tokens=False)\\n        if len(full) <= prefix_length or full[:prefix_length] != prefix_ids:\\n            raise ValueError("The tokenizer does not preserve the MCQ assistant prefix at the answer boundary")\\n        continuations.append(full[prefix_length:])\\n    if len({tuple(ids) for ids in continuations}) != len(MCQ_LETTERS):\\n        raise ValueError("The tokenizer must produce four distinct answer continuations")\\n    return continuations\\n\\n\\ndef prepare_benchmarks(tokenizer, root, languages=("de", "es", "fr", "ja", "zh"),\\n                       budget=100_000, seed=42, max_prompt_tokens=3072,\\n                       max_questions=None, max_shots=None, *, strict_mgsm=True,\\n                       mcq_prefill=True, exclude_rows=None, exclude_prompts=None, dataset_cache=None):\\n    """Create test prompts ONCE; store their exact tokenization and SHA-256.\\n\\n    No dataset is repeated/truncated to fill 100k tokens: 100k is a cap.\\n    Long prompts beyond max_prompt_tokens are skipped and counted.\\n    """\\n    import datasets\\n    dataset_cache = {} if dataset_cache is None else dataset_cache\\n\\n    def load_dataset(name, config, split):\\n        key = (name, config, split)\\n        if key not in dataset_cache:\\n            dataset_cache[key] = datasets.load_dataset(name, config, split=split)\\n        return dataset_cache[key]\\n\\n    root = Path(root)\\n    root.mkdir(parents=True, exist_ok=True)\\n    if budget < 1 or max_prompt_tokens < 128:\\n        raise ValueError("Invalid benchmark token budget or prompt-length bound")\\n    if max_questions is not None and max_questions < 1:\\n        raise ValueError("max_questions must be positive")\\n    if max_shots is not None and max_shots < 1:\\n        raise ValueError("max_shots must be positive")\\n    prompts, summaries = {}, []\\n    mmlu_dev = None\\n    for kind in BENCHMARKS:\\n        default_shots = 8 if kind == "MGSM" else 5\\n        shots = default_shots if max_shots is None else min(default_shots, max_shots)\\n        for lang in languages:\\n            refs = normalized_lang(lang)\\n            if kind == "MMMLU":\\n                if refs["mmmlu"] not in MMMLU_LOCALES:\\n                    print(f"Skip MMMLU/{lang}: language not in MMMLU")\\n                    continue\\n                test = load_dataset("openai/MMMLU", refs["mmmlu"], split="test")\\n                if mmlu_dev is None:\\n                    mmlu_dev = load_dataset("cais/mmlu", "all", split="dev")\\n                demos_by_subject = {}\\n                for row in mmlu_dev:\\n                    subject = str(row["subject"])\\n                    demos_by_subject.setdefault(subject, []).append(row)\\n                def content(row):\\n                    subject = str(row["Subject"])\\n                    demonstrations = demos_by_subject.get(subject, [])[:shots]\\n                    if len(demonstrations) != shots:\\n                        raise ValueError(f"MMLU DEV subject {subject} needs {shots} demonstrations")\\n                    def english_demo(record):\\n                        opts = record["choices"]\\n                        raw_answer = record["answer"]\\n                        answer = (raw_answer.upper() if isinstance(raw_answer, str)\\n                                  and raw_answer.upper() in MCQ_LETTERS\\n                                  else MCQ_LETTERS[int(raw_answer)])\\n                        return "Question: " + str(record["question"]) + "\\\\n" + "\\\\n".join(\\n                            f"{l}. {v}" for l, v in zip(MCQ_LETTERS, opts)) + f"\\\\nAnswer: {answer}\\\\n\\\\n"\\n                    return ("Answer the multiple-choice question with one letter A, B, C, or D.\\\\n\\\\n"\\n                            + "".join(english_demo(d) for d in demonstrations)\\n                            + mcq_example(row, kind, False))\\n            elif kind == "INCLUDE":\\n                test = load_dataset("CohereLabs/include-base-44", refs["include"], split="test")\\n                demos = load_dataset("CohereLabs/include-base-44", refs["include"], split="validation")\\n                demo_rows = list(demos)[:shots]\\n                if len(demo_rows) < shots:\\n                    raise ValueError(f"INCLUDE/{lang} needs {shots} validation demonstrations")\\n                def content(row):\\n                    return ("Answer the multiple-choice question with one letter A, B, C, or D.\\\\n\\\\n"\\n                            + "".join(mcq_example(d, kind, True) for d in demo_rows)\\n                            + mcq_example(row, kind, False))\\n            else:\\n                test = load_dataset("juletxara/mgsm", refs["mgsm"], split="test")\\n                demos = load_dataset("juletxara/mgsm", refs["mgsm"], split="train")\\n                demo_rows = list(demos)[:shots]\\n                if len(demo_rows) < shots:\\n                    raise ValueError(f"MGSM/{lang} needs {shots} training demonstrations")\\n                def content(row):\\n                    return (OLD_MGSM_INSTRUCTION + "\\\\n\\\\n"\\n                            + "".join(mgsm_example(d, True) for d in demo_rows)\\n                            + mgsm_example(row, False))\\n\\n            # Fixed shuffled order avoids dataset row-order/topic bias.\\n            order = list(range(len(test)))\\n            random.Random(seed + sum(map(ord, lang)) + 10000 * BENCHMARKS.index(kind)).shuffle(order)\\n            samples, used, skipped_long, skipped_prior = [], 0, 0, 0\\n            blocked_rows = (exclude_rows or {}).get(kind, ())\\n            blocked_prompts = (exclude_prompts or {}).get((kind, lang), ())\\n            selected_reference_hashes = set()\\n            for idx in order:\\n                if idx in blocked_rows:\\n                    skipped_prior += 1\\n                    continue\\n                row = test[idx]\\n                raw_content = content(row)\\n                raw_prompt = to_chat_prompt(tokenizer, raw_content)\\n                reference_hash = hashlib.sha256(raw_prompt.encode("utf-8")).hexdigest()\\n                if reference_hash in blocked_prompts:\\n                    skipped_prior += 1\\n                    continue\\n                if exclude_rows is not None and reference_hash in selected_reference_hashes:\\n                    continue\\n                if kind == "MGSM" and strict_mgsm:\\n                    prompt = to_chat_prompt(tokenizer, raw_content.replace(OLD_MGSM_INSTRUCTION, MGSM_INSTRUCTION, 1))\\n                else:\\n                    prompt = raw_prompt\\n                assistant_prefill = MCQ_ASSISTANT_PREFILL if kind != "MGSM" and mcq_prefill else ""\\n                scoring_prompt = prompt + assistant_prefill\\n                tokens = tokenizer.encode(scoring_prompt, add_special_tokens=False)\\n                if len(tokens) > max_prompt_tokens:\\n                    skipped_long += 1\\n                    continue\\n                if used + len(tokens) > budget:\\n                    # Skip instead of cutting through a question/prompt.\\n                    continue\\n                ans = (normalize_answer(row, kind) if kind != "MGSM"\\n                       else str(row["answer_number"]))\\n                sample = {\\n                    "benchmark": kind, "language": lang, "source_index": int(idx),\\n                    "prompt": prompt, "prompt_ids": tokens, "answer": ans,\\n                    "assistant_prefill": assistant_prefill,\\n                    "answer_continuations": (mcq_continuations(tokenizer, scoring_prompt, tokens)\\n                                             if assistant_prefill else None),\\n                    "n_prompt_tokens": len(tokens), "reference_prompt_sha256": reference_hash,\\n                    "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),\\n                    "scoring_prompt_sha256": hashlib.sha256(scoring_prompt.encode("utf-8")).hexdigest(),\\n                }\\n                samples.append(sample)\\n                selected_reference_hashes.add(reference_hash)\\n                used += len(tokens)\\n                if used == budget or (max_questions is not None and len(samples) == max_questions):\\n                    break\\n            if max_questions is not None and not samples:\\n                raise RuntimeError(f"No eligible {kind}/{lang} prompts within the token budget")\\n            prompts[(kind, lang)] = samples\\n            summaries.append({\\n                "Benchmark": kind, "Language": lang,\\n                "Prompt_Token_Budget": budget, "Actual_Input_Tokens": used,\\n                "Questions": len(samples), "Available_Test_Questions": len(test),\\n                "Skipped_Too_Long": skipped_long, "Skipped_Prior_Test_Question": skipped_prior,\\n                "Token_Shortfall": budget - used,\\n                "Few_Shot": shots, "Question_Cap": max_questions,\\n                "Prompt_SHA256": hashlib.sha256(\\n                    "".join(x["prompt_sha256"] for x in samples).encode()).hexdigest(),\\n            })\\n            file = root / f"prompts_{kind}_{lang}.jsonl"\\n            with file.open("w", encoding="utf-8") as f:\\n                for sample in samples:\\n                    # Persist prompts without duplicating token IDs into disk.\\n                    f.write(json.dumps({k:v for k,v in sample.items() if k != "prompt_ids"},\\n                                       ensure_ascii=False) + "\\\\n")\\n            print(f"{kind}/{lang}: {len(samples)} questions, {used:,}/{budget:,} input tokens", flush=True)\\n    frame = pd.DataFrame(summaries)\\n    frame.to_csv(root / "prompt_budget.csv", index=False)\\n    return prompts, frame\\n\\n\\ndef parse_mcq_generated_answer(completion):\\n    """Recover an A-D answer from a generated response, or None if ambiguous.\\n\\n    Handles multilingual answer cues, newlines, Markdown and full-width letters.\\n    Never treats an arbitrary A-D appearing in a long explanation as an answer.\\n    This is separate from the primary forced-choice probability scorer.\\n    """\\n    if not isinstance(completion, str) or not completion.strip():\\n        return None\\n    text = unicodedata.normalize("NFKC", completion).strip()\\n    # Qwen3 can occasionally emit thinking tags despite enable_thinking=False.\\n    # If present, only the final answer after the completed thinking block counts.\\n    if "</think>" in text:\\n        text = text.rsplit("</think>", 1)[-1].strip()\\n    elif "<think>" in text:\\n        return None  # No finished answer outside the thinking block.\\n\\n    # Strong explicit answer cues, with optional copulas and whitespace/newlines.\\n    # Stronger cues take priority over a generic \\\'answer\\\' in an explanation.\\n    patterns = [\\n        r"\\\\b(?:the\\\\s+)?(?:correct|right|final)\\\\s+(?:answer|choice|option)\\\\b|\\\\b(?:respuesta\\\\s+final|r[eé]ponse\\\\s+finale|endg[uü]ltige\\\\s+antwort)\\\\b",\\n        r"(?:正解|最終回答|最终答案|最終答案|正しい答え|答え|答案|回答|解答)",\\n        r"\\\\b(?:answer|choice|option|respuesta|r[eé]ponse|antwort|opci[oó]n)\\\\b",\\n    ]\\n    suffix = (r"(?:\\\\s|[:：=,.]|\\\\b(?:is|ist|es|est|serait|would\\\\s+be)\\\\b|"\\n              r"[はがでを的是为]|\\\\(|\\\\*|`)*\\\\s*\\\\(?\\\\*{0,2}([A-D])"\\n              r"(?=$|[^A-Za-z])")\\n    for prefix in patterns:\\n        hits = list(re.finditer("(?:" + prefix + ")" + suffix, text, flags=re.IGNORECASE))\\n        if hits:\\n            labels = {hit.group(1).upper() for hit in hits}\\n            return labels.pop() if len(labels) == 1 else None\\n\\n    # A letter presented as the leading answer line, e.g. \\\'C. ...\\\' or \\\'**B**\\\'.\\n    leading_labels = set()\\n    for line in text.splitlines():\\n        line = line.strip().lstrip(\\\'-•> \\\').strip(\\\'`*_ \\\')\\n        m = re.match(r"^\\\\(?([A-D])\\\\)?(?:\\\\s*$|\\\\s*[.):：]\\\\s*|\\\\s+is\\\\s+correct\\\\b)",\\n                     line, flags=re.IGNORECASE)\\n        if m and len(re.findall(r"(?<![A-Za-z])[A-D]\\\\s*[.):：]\\\\s+", line, flags=re.IGNORECASE)) <= 1:\\n            leading_labels.add(m.group(1).upper())\\n    return leading_labels.pop() if len(leading_labels) == 1 else None\\n\\n\\n@torch.inference_mode()\\ndef mcq_predict(model, prompt_ids, continuations):\\n    """Rank the four continuations after Answer: and measure their probability mass."""\\n    device = next(model.parameters()).device\\n    if all(len(ids) == 1 for ids in continuations):\\n        inputs = torch.tensor([prompt_ids], dtype=torch.long, device=device)\\n        logits = model(input_ids=inputs, use_cache=False, logits_to_keep=1,\\n                       return_dict=True).logits[0, -1]\\n        log_probs = F.log_softmax(logits.float(), dim=-1)\\n        scores = log_probs[[ids[0] for ids in continuations]]\\n    else:\\n        scores = []\\n        for continuation in continuations:\\n            inputs = torch.tensor([prompt_ids + continuation], dtype=torch.long, device=device)\\n            logits = model(input_ids=inputs, use_cache=False, logits_to_keep=len(continuation) + 1,\\n                           return_dict=True).logits[0, :-1]\\n            targets = torch.tensor(continuation, device=device)[:, None]\\n            scores.append(F.log_softmax(logits.float(), dim=-1).gather(-1, targets).sum())\\n        scores = torch.stack(scores)\\n    if not torch.isfinite(scores).all():\\n        raise ArithmeticError("The MCQ scorer received nonfinite answer log probabilities")\\n    return MCQ_LETTERS[int(scores.argmax())], float(scores.exp().sum() * 100)\\n\\n\\n@torch.inference_mode()\\ndef generate_completion(model, prompt_ids, tokenizer, max_new_tokens):\\n    device = next(model.parameters()).device\\n    ids = torch.tensor(prompt_ids, device=device, dtype=torch.long)[None]\\n    eos = model.generation_config.eos_token_id\\n    if eos is None:\\n        eos = tokenizer.eos_token_id\\n    eos_ids = {eos} if isinstance(eos, int) else set(eos or [])\\n    generated = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids),\\n                               use_cache=False, do_sample=False,\\n                               max_new_tokens=max_new_tokens,\\n                               pad_token_id=tokenizer.eos_token_id, eos_token_id=eos)\\n    new_ids = generated[0, ids.shape[1]:].tolist()\\n    completion = tokenizer.decode(new_ids, skip_special_tokens=True)\\n    if new_ids and new_ids[-1] in eos_ids:\\n        stop_reason = "eos"\\n    elif len(new_ids) >= max_new_tokens:\\n        stop_reason = "max_new_tokens"\\n    else:\\n        stop_reason = "other"\\n    return completion, len(new_ids), stop_reason\\n\\n\\ndef evaluate_one(model, tokenizer, prompts, model_name, stage, plan,\\n                 results_csv, max_new_tokens=128, mcq_generation=True,\\n                 mcq_max_new_tokens=64):\\n    rows = []\\n    model.eval()\\n    for (kind, lang), samples in prompts.items():\\n        for sample in samples:\\n            choice_probability_pct = None\\n            mgsm_detail = {}\\n            generated_mcq_letter = generated_mcq_completion = ""\\n            generated_mcq_valid = generated_mcq_correct = generated_mcq_truncated = None\\n            if kind == "MGSM":\\n                generated_text, generated_tokens, stop_reason = generate_completion(\\n                    model, sample["prompt_ids"], tokenizer, max_new_tokens=max_new_tokens)\\n                answer_text, parse_status, outcome = classify(generated_text, lang, sample["answer"], stop_reason)\\n                correct = outcome == "correct"\\n                mgsm_detail = {"MGSM_Generated_Tokens": generated_tokens, "MGSM_Stop_Reason": stop_reason,\\n                               "MGSM_Parse_Status": parse_status, "MGSM_Outcome": outcome}\\n            else:\\n                answer_text, choice_probability_pct = mcq_predict(\\n                    model, sample["prompt_ids"], sample["answer_continuations"])\\n                correct = answer_text == sample["answer"]\\n                generated_text = ""\\n                if mcq_generation:\\n                    generated_mcq_completion, _, stop_reason = generate_completion(\\n                        model, sample["prompt_ids"], tokenizer, max_new_tokens=mcq_max_new_tokens)\\n                    generated_mcq_truncated = stop_reason == "max_new_tokens"\\n                    parsed = None if generated_mcq_truncated else parse_mcq_generated_answer(generated_mcq_completion)\\n                    generated_mcq_letter = parsed or ""\\n                    generated_mcq_valid = parsed is not None\\n                    generated_mcq_correct = parsed == sample["answer"]\\n            rows.append({\\n                "Benchmark": kind, "Language": lang, "Method": model_name,\\n                "Compression_Steps": stage, "Plan": plan,\\n                "Source_Index": sample["source_index"], "Prompt_SHA256": sample["prompt_sha256"],\\n                "Scoring_Prompt_SHA256": sample["scoring_prompt_sha256"],\\n                "Input_Tokens": sample["n_prompt_tokens"],\\n                "MCQ_Choice_Probability_pct": choice_probability_pct,\\n                "Gold": sample["answer"], "Prediction": answer_text,\\n                "Correct": bool(correct), "Generated_Completion": generated_text,\\n                "MCQ_Generated_Prediction": generated_mcq_letter,\\n                "MCQ_Generated_Valid": generated_mcq_valid,\\n                "MCQ_Generated_Correct": generated_mcq_correct,\\n                "MCQ_Generated_Completion": generated_mcq_completion,\\n                "MCQ_Generated_Truncated": generated_mcq_truncated,\\n                **{field: mgsm_detail.get(field) for field in (\\n                    "MGSM_Generated_Tokens", "MGSM_Stop_Reason", "MGSM_Parse_Status", "MGSM_Outcome")},\\n            })\\n        # Incremental crash-safe outputs.\\n        if rows:\\n            frame = pd.DataFrame(rows)\\n            frame.to_csv(results_csv, mode="a", header=not Path(results_csv).exists(), index=False)\\n            rows = []\\n        print(f"  {model_name} stage={stage}: {kind}/{lang} complete", flush=True)\\n\\n\\ndef run_fewshot_evaluation(run_dir, languages=("de", "es", "fr", "ja", "zh"),\\n                           token_budget=100_000, stages=(1, 2, 3, 4, 5),\\n                           methods=nc.METHODS, force=False, max_new_tokens=128,\\n                           max_questions=None, max_shots=None, max_prompt_tokens=3072,\\n                           mcq_generation=True, mcq_max_new_tokens=64,\\n                           *, evaluation_dir=None, prepare=None):\\n    """Re-run original and selected compressed models on EXACTLY the same prompts.\\n\\n    Original checkpoint is loaded from the same model/revision saved at selection.\\n    No new training or benchmark-driven model selection takes place.\\n    """\\n    out = Path(run_dir)\\n    if not (out / "selection_locked.json").exists():\\n        raise RuntimeError("Train/selection must finish before benchmark evaluation")\\n    selection = json.loads((out / "selection_locked.json").read_text())\\n    for file, digest in selection["file_sha256"].items():\\n        if hashlib.sha256((out / file).read_bytes()).hexdigest() != digest:\\n            raise RuntimeError(f"Selection lock mismatch: {file}")\\n    cfg = nc.Config(**json.loads((out / "configuration.json").read_text()))\\n    env = json.loads((out / "environment.json").read_text())\\n    for method in methods:\\n        if method not in nc.METHODS:\\n            raise ValueError(f"Unknown compression method {method}")\\n    final_stage = int(selection.get("max_merges", cfg.max_merges))\\n    if any(stage < 1 or stage > final_stage for stage in stages):\\n        raise ValueError(f"Stages must be within 1..{final_stage}")\\n\\n    import torch\\n    device = "cuda" if torch.cuda.is_available() else "cpu"\\n    tokenizer = nc.sm.AutoTokenizer.from_pretrained(cfg.model_id, revision=env["model_resolved_revision"])\\n    teacher = nc.sm.AutoModelForCausalLM.from_pretrained(\\n        cfg.model_id, revision=env["model_resolved_revision"],\\n        attn_implementation="eager", dtype=torch.float32,\\n    ).to(device).eval().requires_grad_(False)\\n    teacher.config.use_cache = teacher.generation_config.use_cache = False\\n    if nc.parameter_digest(teacher) != env["teacher_parameter_sha256"]:\\n        raise ValueError("Loaded original Qwen3 differs from the training original")\\n\\n    evaluation_dir = Path(evaluation_dir) if evaluation_dir is not None else out / "fewshot_benchmarks"\\n    evaluation_dir.mkdir(exist_ok=True)\\n    csv = evaluation_dir / "fewshot_predictions.csv"\\n    if csv.exists() and not force:\\n        raise RuntimeError("Benchmark results already exist; use force=True to overwrite intentionally")\\n    if csv.exists():\\n        csv.unlink()\\n    if prepare is None:\\n        prompts, budget_table = prepare_benchmarks(\\n            tokenizer, evaluation_dir, languages=languages, budget=token_budget, seed=cfg.seed,\\n            max_questions=max_questions, max_shots=max_shots, max_prompt_tokens=max_prompt_tokens)\\n    else:\\n        prompts, budget_table = prepare(tokenizer, evaluation_dir)\\n    # Original baseline is run first and is never modified or re-trained.\\n    evaluate_one(teacher, tokenizer, prompts, "Original", 0, "none", csv, max_new_tokens,\\n                 mcq_generation=mcq_generation, mcq_max_new_tokens=mcq_max_new_tokens)\\n    for stage in stages:\\n        for method in methods:\\n            student, groups = nc.replay_selected_stage(out, method, stage, teacher=teacher)\\n            student.config.use_cache = student.generation_config.use_cache = False\\n            plan = nc.plan_string(groups, one_based=bool(cfg.fixed_layer_pairs))\\n            evaluate_one(student, tokenizer, prompts, method, stage, plan, csv, max_new_tokens,\\n                         mcq_generation=mcq_generation, mcq_max_new_tokens=mcq_max_new_tokens)\\n            del student\\n            nc.cleanup()\\n    pred = pd.read_csv(csv, keep_default_na=False)\\n    scores = pred.groupby(["Benchmark", "Language", "Method", "Compression_Steps", "Plan"],\\n                          as_index=False).agg(Accuracy=("Correct", "mean"),\\n                                               Correct=("Correct", "sum"),\\n                                               Questions=("Correct", "size"),\\n                                               Input_Tokens=("Input_Tokens", "sum"))\\n    baseline = scores[scores.Method == "Original"][["Benchmark", "Language", "Accuracy"]]\\n    baseline = baseline.rename(columns={"Accuracy": "Original_Accuracy"})\\n    scores = scores.merge(baseline, on=["Benchmark", "Language"], how="left")\\n    scores["Accuracy_Delta_pp"] = 100 * (scores["Accuracy"] - scores["Original_Accuracy"])\\n    scores.to_csv(evaluation_dir / "fewshot_accuracy.csv", index=False)\\n    mcq = pred.loc[pred["Benchmark"].isin(("MMMLU", "INCLUDE"))].copy()\\n    mcq["MCQ_Choice_Probability_pct"] = pd.to_numeric(mcq["MCQ_Choice_Probability_pct"])\\n    for letter in MCQ_LETTERS:\\n        mcq[letter] = mcq["Prediction"].eq(letter)\\n    choice_counts = mcq.groupby(["Benchmark", "Language", "Method", "Compression_Steps"], as_index=False).agg(\\n        Questions=("Prediction", "size"),\\n        **{letter: (letter, "sum") for letter in MCQ_LETTERS},\\n        Median_Choice_Probability_pct=("MCQ_Choice_Probability_pct", "median"),\\n    )\\n    choice_counts["All_Same_Letter"] = choice_counts[list(MCQ_LETTERS)].max(axis=1).eq(choice_counts["Questions"])\\n    choice_counts.to_csv(evaluation_dir / "fewshot_choice_counts.csv", index=False)\\n    mgsm = pred.loc[pred["Benchmark"] == "MGSM"]\\n    outcomes = mgsm.groupby(["Language", "Method", "MGSM_Outcome"]).size().rename("Questions").reset_index()\\n    outcomes.to_csv(evaluation_dir / "mgsm_outcomes.csv", index=False)\\n    if mcq_generation:\\n        # Report generated answers separately; do NOT overwrite forced-choice accuracy.\\n        mcq["MCQ_Generated_Valid"] = mcq["MCQ_Generated_Valid"].astype(str).str.lower().eq("true")\\n        mcq["MCQ_Generated_Correct"] = mcq["MCQ_Generated_Correct"].astype(str).str.lower().eq("true")\\n        generated_scores = mcq.groupby(\\n            ["Benchmark", "Language", "Method", "Compression_Steps", "Plan"],\\n            as_index=False,\\n        ).agg(\\n            Generated_Accuracy=("MCQ_Generated_Correct", "mean"),\\n            Generated_Correct=("MCQ_Generated_Correct", "sum"),\\n            Generated_Valid=("MCQ_Generated_Valid", "sum"),\\n            Questions=("MCQ_Generated_Correct", "size"),\\n        )\\n        generated_scores["Generated_Valid_Rate"] = (\\n            generated_scores["Generated_Valid"] / generated_scores["Questions"]\\n        )\\n        generated_scores.to_csv(evaluation_dir / "fewshot_generated_accuracy.csv", index=False)\\n        print("Saved MCQ free-generation diagnostic to fewshot_generated_accuracy.csv")\\n    print(f"Saved {len(scores)} summary rows to {evaluation_dir}")\\n    return scores, budget_table\\n\', \'verify_qwen3.py\': \'\\n"""Small randomly initialized Qwen3 checks, no downloads or performance claims."""\\nfrom __future__ import annotations\\n\\nimport copy\\nimport numpy as np\\nimport torch\\nfrom transformers import Qwen3Config, Qwen3ForCausalLM\\n\\nimport qwen3_compression_multi as nc\\n\\n\\ndef run_checks():\\n    nc.sm.check_transformers_version()\\n    nc.sm.set_seed(321)\\n    config = Qwen3Config(\\n        vocab_size=128, hidden_size=64, intermediate_size=128,\\n        num_hidden_layers=5, num_attention_heads=4, num_key_value_heads=2,\\n        head_dim=16, max_position_embeddings=128,\\n        attention_dropout=0.0, use_cache=False,\\n    )\\n    config._attn_implementation = "eager"\\n    teacher = Qwen3ForCausalLM(config).eval().requires_grad_(False)\\n    teacher.generation_config.use_cache = False\\n    groups = nc.initial_groups(5)\\n    assert nc.plan_string(nc.merge_groups(nc.merge_groups(groups, 1), 1)) == "1-3"\\n    assert nc.plan_string(nc.merge_groups(nc.merge_groups(groups, 1), 2)) == "1-2;3-4"\\n    successes = [{"check": "disjoint_and_chained_are_distinct", "status": "PASS"}]\\n    pilot = nc.make_config("pilot")\\n    nc.validate_config(pilot)\\n    pilot_groups = nc.initial_groups(28)\\n    for pair in pilot.fixed_layer_pairs:\\n        boundary = nc.fixed_pair_boundary(pilot_groups, pair)\\n        pilot_groups = nc.merge_groups(pilot_groups, boundary)\\n    assert len(pilot_groups) == 26 and (3,) in pilot_groups\\n    assert nc.plan_string(pilot_groups, one_based=True) == "7-8;13-14"\\n    successes.append({"check": "fixed_pairs_keep_original_layer_numbers", "status": "PASS"})\\n\\n    cfg = nc.make_config(mode="quick", context_length=16, eval_stride=8,\\n                         local_tokens=32, kd_tokens=32, holdout_tokens=16,\\n                         max_merges=2)\\n    ids = torch.randint(0, 128, (1, 16))\\n    original = teacher(ids, use_cache=False).logits\\n    assert original.shape == (1, 16, 128)\\n    for method in nc.METHODS:\\n        affine = (np.eye(64), np.zeros(64)) if method == "LinearMultiply" else None\\n        student, merged_groups, span = nc.build_next_student(\\n            teacher, teacher, groups, 1, method, cfg, affine=affine)\\n        assert len(student.model.layers) == 4\\n        out = student(ids, use_cache=False).logits\\n        assert out.shape == original.shape and out.isfinite().all()\\n        assert merged_groups[1] == (1, 2)\\n        assert span == (1, 2)\\n        assert hasattr(student.model.layers[1], "attention_type"), (method, "attention_type missing")\\n        assert student.model.layers[1].attention_type == teacher.model.layers[1].attention_type\\n        expected = {id(p) for p in student.model.layers[1].parameters()} if method == "NonlinearMultiply" else set()\\n        actual = {id(p) for p in student.parameters() if p.requires_grad}\\n        assert actual == expected\\n        if method == "NonlinearMultiply":\\n            loss, _ = nc.local_objective_span(teacher, student, 1, span, ids, cfg)\\n            loss.backward()\\n            assert any(p.grad is not None for p in student.model.layers[1].parameters())\\n        successes.append({"check": f"{method}_qwen_forward_and_parameter_freeze", "status": "PASS"})\\n        # Stage 2 also exercises a wrapper generated by stage 1, including\\n        # Qwen3Model.forward\\\'s per-layer attention-mask dispatch.\\n        student2, groups2, span2 = nc.build_next_student(\\n            student, teacher, merged_groups, 1, method, cfg, affine=affine)\\n        assert groups2[1] == (1, 2, 3) and span2 == (1, 3)\\n        assert len(student2.model.layers) == 3\\n        assert student2.model.layers[1].attention_type == teacher.model.layers[1].attention_type\\n        out2 = student2(ids, use_cache=False).logits\\n        assert out2.shape == original.shape and out2.isfinite().all()\\n        successes.append({"check": f"{method}_second_merge_qwen_forward", "status": "PASS"})\\n        del student2, student\\n    return successes\\n\', \'strict_mgsm.py\': \'"""Parse a single explicit MGSM final answer and retain the generation outcome."""\\n\\nimport re\\nimport unicodedata\\nfrom decimal import Decimal, InvalidOperation\\n\\n\\nFINAL_LINE = re.compile(r\\\'(?:\\\\*\\\\*|__)?\\\\s*final\\\\s+answer\\\\s*:\\\\s*(?:\\\\*\\\\*|__)?\\\\s*(.*)\\\', re.IGNORECASE)\\n\\n\\ndef parse_number(text, language):\\n    value = unicodedata.normalize(\\\'NFKC\\\', str(text)).strip().replace(\\\'−\\\', \\\'-\\\')\\n    value = value.removesuffix(\\\'.\\\').rstrip()\\n    if value.startswith((\\\'**\\\', \\\'__\\\')) and value.endswith(value[:2]):\\n        value = value[2:-2].strip()\\n    if value.startswith(\\\'$\\\') and value.endswith(\\\'$\\\'):\\n        value = value[1:-1].strip()\\n    boxed = re.fullmatch(r\\\'\\\\\\\\boxed\\\\{([^{}]+)\\\\}\\\', value)\\n    if boxed:\\n        value = boxed[1].strip()\\n    if not re.fullmatch(r\\\'[+-]?\\\\d+(?:[., ]\\\\d+)*\\\', value):\\n        return None\\n    sign = value[0] if value[0] in \\\'+-\\\' else \\\'\\\'\\n    digits = value[len(sign):]\\n    punctuation = {sep for sep in \\\'.,\\\' if sep in digits}\\n    decimal = None\\n    if len(punctuation) == 2:\\n        decimal = \\\'.\\\' if digits.rindex(\\\'.\\\') > digits.rindex(\\\',\\\') else \\\',\\\'\\n    elif punctuation:\\n        sep = next(iter(punctuation))\\n        grouping = \\\'.\\\' if language in (\\\'de\\\', \\\'es\\\') else \\\',\\\' if language in (\\\'ja\\\', \\\'zh\\\', \\\'en\\\') else \\\' \\\'\\n        is_grouped = re.fullmatch(rf\\\'[1-9]\\\\d{{0,2}}(?:{re.escape(sep)}\\\\d{{3}})+\\\', digits)\\n        if not (is_grouped and (sep == grouping or digits.count(sep) > 1)):\\n            decimal = sep\\n\\n    if decimal:\\n        whole, _, fraction = digits.rpartition(decimal)\\n        if decimal in whole or not fraction.isdecimal():\\n            return None\\n    else:\\n        whole, fraction = digits, \\\'\\\'\\n    separators = {sep for sep in \\\'., \\\' if sep in whole}\\n    if separators:\\n        if len(separators) != 1:\\n            return None\\n        sep = next(iter(separators))\\n        if not re.fullmatch(rf\\\'[1-9]\\\\d{{0,2}}(?:{re.escape(sep)}\\\\d{{3}})+\\\', whole):\\n            return None\\n        whole = whole.replace(sep, \\\'\\\')\\n    try:\\n        return Decimal(sign + whole + (\\\'.\\\' + fraction if fraction else \\\'\\\'))\\n    except InvalidOperation:\\n        return None\\n\\n\\ndef parse_final_answer(completion, language):\\n    lines = [line.strip() for line in unicodedata.normalize(\\\'NFKC\\\', completion).splitlines() if line.strip()]\\n    lines = [line[2:-2].strip() if line.startswith((\\\'**\\\', \\\'__\\\')) and line.endswith(line[:2])\\n             and line.count(line[:2]) == 2 else line for line in lines]\\n    matches = [(index, match) for index, line in enumerate(lines) if (match := FINAL_LINE.fullmatch(line))]\\n    if not matches:\\n        return None, \\\'missing_final_answer\\\'\\n    index, match = matches[-1]\\n    if index != len(lines) - 1:\\n        return None, \\\'final_answer_not_last\\\'\\n    answer = parse_number(match[1], language)\\n    return (answer, \\\'valid\\\') if answer is not None else (None, \\\'invalid_final_answer\\\')\\n\\n\\ndef gold_number(text):\\n    """The dataset supplies a canonical numeric answer, independent of language."""\\n    return Decimal(str(text))\\n\\n\\ndef classify(completion, language, gold, stop_reason):\\n    answer, parse_status = parse_final_answer(completion, language)\\n    if stop_reason == \\\'max_new_tokens\\\':\\n        result = \\\'truncated\\\'\\n    elif answer is None:\\n        result = parse_status\\n    else:\\n        result = \\\'correct\\\' if answer == gold_number(gold) else \\\'wrong_answer\\\'\\n    return (str(answer) if answer is not None else \\\'\\\'), parse_status, result\\n\', \'fresh_evaluation_protocol.py\': \'"""Audit old MGSM completions and evaluate the same checkpoints on new questions."""\\n\\nimport tempfile\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\nimport qwen3_benchmarks as be\\nfrom strict_mgsm import classify\\n\\n\\nHISTORY = [(\\\'pilot_15\\\', 15, 2048, False, {(\\\'MMMLU\\\', \\\'de\\\'): \\\'647369c0aec83645a7d646525950242997418e30e3ec1c94922155bd7a0f2f58\\\', (\\\'MMMLU\\\', \\\'ja\\\'): \\\'61b0d34a25c28b41083ef55a10077d5c0b121a9de08520a806007ecea8db810c\\\', (\\\'INCLUDE\\\', \\\'de\\\'): \\\'1a43ca196e48a37771e937d73a9a23046b0492bafa834fb591ffce998a407974\\\', (\\\'INCLUDE\\\', \\\'ja\\\'): \\\'40132d371556348d1f92d69029255585dc7ee81bff1901230a7030afd41e2e6b\\\', (\\\'MGSM\\\', \\\'de\\\'): \\\'d398b4f90ebe656cc00367846e6660543aa16854a4e246de16fb7369b9b7c963\\\', (\\\'MGSM\\\', \\\'ja\\\'): \\\'9dca5125ab37db747f57a4e2249e3ef84b236b140e7c5f51632d45b930cc8611\\\'}), (\\\'pilot_12\\\', 12, 1024, True, {(\\\'MMMLU\\\', \\\'de\\\'): \\\'5e4c404bc099e94dd45c8b74359df4884af7085a01c747f8ace19c5e58d4bdff\\\', (\\\'MMMLU\\\', \\\'ja\\\'): \\\'7298de6ba5de82c13574d015a6aa349d98af4f318326d6bb7ba56d0f9513a427\\\', (\\\'INCLUDE\\\', \\\'de\\\'): \\\'954b873e7665b452ed39aac7a351adb2ba749acd52224156a70d702347a43995\\\', (\\\'INCLUDE\\\', \\\'ja\\\'): \\\'9b1716a94689578cd41b0e86473c73ecef5c11bfcab73f7233ff4d8192281fe9\\\', (\\\'MGSM\\\', \\\'de\\\'): \\\'af908375c85dd8e4d6a36af8edc071d8a984203cc21f33e93427d50a23daaedf\\\', (\\\'MGSM\\\', \\\'ja\\\'): \\\'fec3d584ff87bd4c1de9701ccafa1f287df68b1ce3707e62334def4cf161a71a\\\'}), (\\\'pilot_40_three_languages\\\', 40, 2048, True, {(\\\'MMMLU\\\', \\\'de\\\'): \\\'01cda2f16d83ae4057c94572fc2498e204b0bcb3310f61c47be47277a6637f2d\\\', (\\\'MMMLU\\\', \\\'ja\\\'): \\\'1c5f565b668b3d8511b7caf5d8a684cc02749b6ac0abe5456bba6d5c6f1c4486\\\', (\\\'MMMLU\\\', \\\'fr\\\'): \\\'584ecdc4cdfbeccec49d1ebab896a72594cff617e7565f7b0bcb63879190ef90\\\', (\\\'INCLUDE\\\', \\\'de\\\'): \\\'28c73c1f6db849a005322f88ff4a348336255c502c6e3e157d68f1104e9e29a1\\\', (\\\'INCLUDE\\\', \\\'ja\\\'): \\\'a42d51066e60c8964208e1f6d531378f5807ce27ff903de1eb2e1a0d6fb93b78\\\', (\\\'INCLUDE\\\', \\\'fr\\\'): \\\'bcfdb488cd1f9e3dd3365ff51d84cabdbc91f24d6ba88f5f9fa796feabc258f8\\\', (\\\'MGSM\\\', \\\'de\\\'): \\\'a8317c02d2ca7b2f63c91e8107726ab24c0b335927af7f77a80a7b96c2ebfb05\\\', (\\\'MGSM\\\', \\\'ja\\\'): \\\'00e5da4f75457e8eaf06b6ad5239f7f9885bd9d38b2a8894eb1ea461abb01a5c\\\', (\\\'MGSM\\\', \\\'fr\\\'): \\\'00bd81483a2dd1b09463bedcef99e7b1e1193d17b8a03d4a9f6cac4a9dfe94bc\\\'})]\\nLANGUAGES = (\\\'de\\\', \\\'ja\\\')\\nMETHOD = \\\'NonlinearMultiply\\\'\\n\\n\\ndef audit_mgsm(previous, output):\\n    fields = [\\\'Benchmark\\\', \\\'Language\\\', \\\'Method\\\', \\\'Compression_Steps\\\', \\\'Source_Index\\\',\\n              \\\'Gold\\\', \\\'Prediction\\\', \\\'Correct\\\', \\\'Generated_Completion\\\']\\n    fields += [key for key in (\\\'Archived_Run\\\', \\\'MGSM_Stop_Reason\\\', \\\'MGSM_Truncated\\\') if key in previous]\\n    audit = previous.loc[previous[\\\'Benchmark\\\'] == \\\'MGSM\\\', fields].copy()\\n    if audit.empty:\\n        raise ValueError(\\\'The archived predictions contain no MGSM completions to audit.\\\')\\n    audit = audit.rename(columns={\\\'Prediction\\\': \\\'Prior_Prediction\\\', \\\'Correct\\\': \\\'Prior_Correct\\\'})\\n    audit[\\\'Prior_Correct\\\'] = audit[\\\'Prior_Correct\\\'].astype(str).str.lower().eq(\\\'true\\\')\\n    reasons = audit.get(\\\'MGSM_Stop_Reason\\\', pd.Series(\\\'unknown\\\', index=audit.index)).fillna(\\\'unknown\\\')\\n    reasons = reasons.replace(\\\'\\\', \\\'unknown\\\')\\n    if \\\'MGSM_Truncated\\\' in audit:\\n        reasons = reasons.mask(audit[\\\'MGSM_Truncated\\\'].astype(str).str.lower().eq(\\\'true\\\'), \\\'max_new_tokens\\\')\\n    audit[\\\'Historical_Stop_Reason\\\'] = reasons\\n    parsed = [classify(row.Generated_Completion, row.Language, row.Gold, row.Historical_Stop_Reason)\\n              for row in audit.itertuples(index=False)]\\n    audit[[\\\'Strict_Prediction\\\', \\\'Strict_Parse_Status\\\', \\\'Strict_Text_Outcome\\\']] = pd.DataFrame(parsed, index=audit.index)\\n    audit[\\\'Strict_Text_Correct\\\'] = audit[\\\'Strict_Text_Outcome\\\'].eq(\\\'correct\\\')\\n    audit[\\\'Scoring_Changed\\\'] = audit[\\\'Prior_Correct\\\'].ne(audit[\\\'Strict_Text_Correct\\\'])\\n    audit.to_csv(output / \\\'prior_mgsm_audit.csv\\\', index=False)\\n\\n\\ndef prepare_fresh(tokenizer, output, previous, seed, questions, max_prompt_tokens,\\n                  *, languages=LANGUAGES, shots=3, token_budget=None):\\n    cache = {}\\n    blocked_rows = defaultdict(set)\\n    blocked_prompts = defaultdict(set)\\n    evidence = []\\n    for row in previous.itertuples(index=False):\\n        blocked_rows[row.Benchmark].add(int(row.Source_Index))\\n        blocked_prompts[row.Benchmark, row.Language].add(row.Prompt_SHA256)\\n\\n    with tempfile.TemporaryDirectory(dir=output) as scratch:\\n        for name, cap, prompt_limit, prefill, expected in HISTORY:\\n            old, summary = be.prepare_benchmarks(\\n                tokenizer, Path(scratch) / name,\\n                languages=tuple(dict.fromkeys(lang for _, lang in expected)), budget=cap * prompt_limit,\\n                seed=42, max_prompt_tokens=prompt_limit, max_questions=cap, max_shots=3,\\n                strict_mgsm=False, mcq_prefill=prefill, dataset_cache=cache,\\n            )\\n            actual = {(row.Benchmark, row.Language): row.Prompt_SHA256 for row in summary.itertuples(index=False)}\\n            mismatches = sorted(key for key in expected if actual.get(key) != expected[key])\\n            if mismatches or actual.keys() != expected.keys() or any(len(group) != cap for group in old.values()):\\n                raise ValueError(f\\\'Cannot verify the earlier {name} sample for {mismatches}. \\\'\\n                                 \\\'Dataset rows or prompt text may have changed. Fresh evaluation stopped.\\\')\\n            for (benchmark, language), samples in old.items():\\n                blocked_rows[benchmark].update(item[\\\'source_index\\\'] for item in samples)\\n                blocked_prompts[benchmark, language].update(item[\\\'prompt_sha256\\\'] for item in samples)\\n                evidence.append({\\\'Historical_Set\\\': name, \\\'Benchmark\\\': benchmark, \\\'Language\\\': language,\\n                                 \\\'Questions\\\': len(samples), \\\'Prompt_SHA256\\\': actual[benchmark, language]})\\n\\n    prompts, budget = be.prepare_benchmarks(\\n        tokenizer, output, languages=languages, budget=token_budget or questions * max_prompt_tokens,\\n        seed=seed, max_prompt_tokens=max_prompt_tokens, max_questions=questions, max_shots=shots,\\n        exclude_rows=blocked_rows, exclude_prompts=blocked_prompts, dataset_cache=cache,\\n    )\\n    if set(prompts) != {(benchmark, language) for benchmark in be.BENCHMARKS for language in languages}:\\n        raise ValueError(\\\'The fresh sample is missing a benchmark-language group.\\\')\\n    for key, group in prompts.items():\\n        benchmark, language = key\\n        if len(group) != questions:\\n            raise ValueError(f\\\'Only {len(group)} new questions fit for {benchmark}/{language}; requested {questions}.\\\')\\n        if any(item[\\\'source_index\\\'] in blocked_rows[benchmark]\\n               or item[\\\'reference_prompt_sha256\\\'] in blocked_prompts[key] for item in group):\\n            raise AssertionError(f\\\'Prior question reused in {key}.\\\')\\n    pd.DataFrame(evidence).to_csv(output / \\\'verified_prior_samples.csv\\\', index=False)\\n    manifest = [{\\\'Benchmark\\\': benchmark, \\\'Excluded_Source_Index\\\': index}\\n                for benchmark, indices in sorted(blocked_rows.items()) for index in sorted(indices)]\\n    pd.DataFrame(manifest).to_csv(output / \\\'excluded_question_indices.csv\\\', index=False)\\n    budget[\\\'Excluded_Prior_Row_Indices\\\'] = budget[\\\'Benchmark\\\'].map(lambda name: len(blocked_rows[name]))\\n    budget[\\\'Verified_Overlap\\\'] = 0\\n    budget.to_csv(output / \\\'prompt_budget.csv\\\', index=False)\\n    print(\\\'Verified historical prompt hashes and zero prior-row overlap in every fresh group.\\\', flush=True)\\n    return prompts, budget\\n\\n\\ndef verify_paired_predictions(output):\\n    pred = pd.read_csv(output / \\\'fewshot_predictions.csv\\\', keep_default_na=False)\\n    keys = [\\\'Benchmark\\\', \\\'Language\\\', \\\'Source_Index\\\', \\\'Scoring_Prompt_SHA256\\\']\\n    groups = pred.groupby(keys)\\n    if not (groups.size().eq(2).all() and groups[\\\'Method\\\'].nunique().eq(2).all()\\n            and groups[\\\'Gold\\\'].nunique().eq(1).all()):\\n        raise ValueError(\\\'Original and merged predictions did not match question for question.\\\')\\n\\n\', \'mathoctopus_training.py\': \'"""Verified MathOctopus training windows for the Hiroyoshi compression experiment."""\\n\\nimport hashlib\\nimport re\\nimport unicodedata\\nfrom dataclasses import replace\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\n\\n\\nMATH_DATASET = \\\'Mathoctopus/GSM8KInstruct_Parallel\\\'\\nMATH_REVISION = \\\'dd8076b83bacd7c6ade2829317e8a500c6ffbec4\\\'\\nGSM_DATASET = \\\'openai/gsm8k\\\'\\nGSM_REVISION = \\\'740312add88f781978c0658806c59bc2815b9866\\\'\\nLANGUAGES = {\\\'English\\\': \\\'en\\\', \\\'Bengali\\\': \\\'bn\\\', \\\'German\\\': \\\'de\\\', \\\'Spanish\\\': \\\'es\\\', \\\'French\\\': \\\'fr\\\',\\n             \\\'Japanese\\\': \\\'ja\\\', \\\'Russian\\\': \\\'ru\\\', \\\'Swahili\\\': \\\'sw\\\', \\\'Thai\\\': \\\'th\\\', \\\'Chinese\\\': \\\'zh\\\'}\\nPROMPT = re.compile(r\\\'Please answer in ([A-Za-z]+)\\\\.\\\\s*### Instruction:\\\\s*(.*?)\\\\s*### Response:\\\\s*\\\\Z\\\', re.S)\\nGSM_ANSWER = re.compile(r\\\'[+-]?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)(?:\\\\.[0-9]+)?\\\')\\nSUBSETS = (\\\'local\\\', \\\'kd\\\', \\\'holdout\\\')\\n\\n\\ndef question_key(text):\\n    text = unicodedata.normalize(\\\'NFKC\\\', text).translate(str.maketrans(\\\'‘’“”\\\', "\\\'\\\'\\\\"\\\\""))\\n    return \\\' \\\'.join(text.casefold().split())\\n\\n\\ndef official_answer(text):\\n    _, delimiter, tail = text.rpartition(\\\'####\\\')\\n    number = tail.strip()\\n    return number.replace(\\\',\\\', \\\'\\\') if delimiter and GSM_ANSWER.fullmatch(number) else None\\n\\n\\ndef tokenize_math_response(tokenizer, question, solution, answer):\\n    prompt = tokenizer.apply_chat_template(\\n        [{\\\'role\\\': \\\'user\\\', \\\'content\\\': (\\n            "Solve the math problem. Show your reasoning and end with \\\'Final answer: <number>\\\'.\\\\n\\\\n"\\n            f\\\'Question: {question}\\\\nSolution:\\\'\\n        )}],\\n        tokenize=False, add_generation_prompt=True, enable_thinking=False,\\n    )\\n    solution = re.sub(r\\\'\\\\s*####\\\\s*[+-]?[0-9][0-9,.]*\\\\s*$\\\', \\\'\\\', solution).strip()\\n    response = f\\\'{solution}\\\\nFinal answer: {answer}\\\' if solution else f\\\'Final answer: {answer}\\\'\\n    encoded = tokenizer(prompt + response, add_special_tokens=False, return_offsets_mapping=True)\\n    response_start = next((i for i, (_, end) in enumerate(encoded[\\\'offset_mapping\\\']) if end > len(prompt)), None)\\n    tokens = encoded[\\\'input_ids\\\'] + [tokenizer.eos_token_id]\\n    if response_start is None or not 0 < response_start < len(tokens) - 1:\\n        raise ValueError(\\\'Expected both prompt tokens and supervised math response tokens.\\\')\\n    return tokens, response_start\\n\\n\\ndef verified_groups(rows, official_train, official_test):\\n    """Keep complete parallel groups anchored to an exact GSM8K TRAIN question."""\\n    groups = {}\\n    active = []\\n    key = None\\n\\n    def commit():\\n        if key is not None and [item[0] for item in active] == list(LANGUAGES.values()):\\n            groups.setdefault(key, {lang: (index, question, solution) for lang, index, question, solution in active})\\n\\n    for index, row in enumerate(rows):\\n        prompt = row.get(\\\'prompt\\\')\\n        match = PROMPT.search(prompt) if isinstance(prompt, str) else None\\n        if match is None or match[1] not in LANGUAGES:\\n            key = None\\n            continue\\n        lang, question = LANGUAGES[match[1]], match[2].strip()\\n        if lang == \\\'en\\\':\\n            commit()\\n            candidate = question_key(question)\\n            key = candidate if candidate in official_train and candidate not in official_test else None\\n            active = []\\n        if key is not None:\\n            solution = row.get(\\\'chosen\\\')\\n            active.append((lang, index, question, solution.strip() if isinstance(solution, str) else \\\'\\\'))\\n    commit()\\n    return groups\\n\\n\\ndef load_math_windows(tokenizer, cfg, counts):\\n    import datasets\\n\\n    train = datasets.load_dataset(GSM_DATASET, \\\'main\\\', split=\\\'train\\\', revision=GSM_REVISION)\\n    test = datasets.load_dataset(GSM_DATASET, \\\'main\\\', split=\\\'test\\\', revision=GSM_REVISION)\\n    train_answers = {question_key(row[\\\'question\\\']): official_answer(row[\\\'answer\\\']) for row in train}\\n    train_answers = {key: answer for key, answer in train_answers.items() if answer is not None}\\n    test_keys = {question_key(row[\\\'question\\\']) for row in test}\\n    parallel = datasets.load_dataset(MATH_DATASET, split=\\\'train\\\', revision=MATH_REVISION)\\n    groups = verified_groups(parallel, train_answers, test_keys)\\n    if not groups:\\n        raise ValueError(\\\'No complete MathOctopus groups matched the official GSM8K training split.\\\')\\n\\n    order = sorted(groups)\\n    np.random.default_rng(cfg.seed + 4049).shuffle(order)\\n    remaining = {lang: iter(order) for lang in cfg.math_train_languages}\\n    used = set()\\n    parts = {name: [] for name in SUBSETS}\\n    response_starts = {name: [] for name in SUBSETS}\\n    manifest = []\\n    for name in SUBSETS:\\n        for _ in range(counts[name]):\\n            lang = cfg.math_train_languages[len(manifest) % len(cfg.math_train_languages)]\\n            for key in remaining[lang]:\\n                if key in used:\\n                    continue\\n                index, question, solution = groups[key][lang]\\n                if not question or not solution or solution.casefold() == "i don\\\'t know.":\\n                    continue\\n                tokens, response_start = tokenize_math_response(tokenizer, question, solution, train_answers[key])\\n                if not 2 <= len(tokens) <= cfg.math_max_window_tokens:\\n                    continue\\n                parts[name].append(torch.tensor(tokens, dtype=torch.long))\\n                response_starts[name].append(response_start)\\n                used.add(key)\\n                manifest.append({\\\'Split\\\': name, \\\'Dataset\\\': MATH_DATASET, \\\'Language\\\': lang, \\\'Tokens\\\': len(tokens),\\n                                 \\\'Source_Row\\\': index, \\\'Problem_SHA256\\\': hashlib.sha256(key.encode(\\\'utf-8\\\')).hexdigest(),\\n                                 \\\'Response_Start\\\': response_start, \\\'Final_Answer\\\': train_answers[key]})\\n                break\\n            else:\\n                raise ValueError(f\\\'Insufficient complete verified {lang} math examples for {name}. \\\'\\n                                 \\\'Increase the math window limit or reduce the math fraction.\\\')\\n    provenance = {\\n        \\\'dataset\\\': MATH_DATASET, \\\'revision\\\': MATH_REVISION, \\\'split\\\': \\\'train\\\',\\n        \\\'canonical_dataset\\\': GSM_DATASET, \\\'canonical_revision\\\': GSM_REVISION,\\n        \\\'filter\\\': \\\'Complete parallel groups with exact normalized English TRAIN match and no TEST match\\\',\\n        \\\'verified_groups\\\': len(groups), \\\'selected_unique_problems\\\': len(used),\\n        \\\'languages\\\': list(cfg.math_train_languages), \\\'max_window_tokens\\\': cfg.math_max_window_tokens,\\n    }\\n    return parts, response_starts, pd.DataFrame(manifest), provenance\\n\\n\\ndef split_mixed_train(tokenizer, wiki_ids, cfg, split_wiki):\\n    counts = {name: getattr(cfg, f\\\'{name}_tokens\\\') // cfg.context_length for name in SUBSETS}\\n    math_counts = {name: int(count * cfg.math_train_fraction + 0.5) for name, count in counts.items()}\\n    if any(not 0 < math_counts[name] < counts[name] for name in SUBSETS):\\n        raise ValueError(\\\'Each mixed subset needs at least one window from each dataset. \\\'\\n                         \\\'Adjust the math fraction or increase its window budget.\\\')\\n    wiki_cfg = replace(cfg, **{f\\\'{name}_tokens\\\': (counts[name] - math_counts[name]) * cfg.context_length\\n                               for name in SUBSETS})\\n    wiki_parts, wiki_manifest = split_wiki(wiki_ids, wiki_cfg)\\n    wiki_manifest = wiki_manifest.assign(Dataset=\\\'google/wiki40b\\\', Tokens=wiki_manifest[\\\'End\\\'] - wiki_manifest[\\\'Start\\\'])\\n    math_parts, response_starts, math_manifest, provenance = load_math_windows(tokenizer, cfg, math_counts)\\n    parts = {name: wiki_parts[name] + math_parts[name] for name in SUBSETS}\\n    parts[\\\'response_starts\\\'] = {\\n        name: [None] * len(wiki_parts[name]) + response_starts[name] for name in (\\\'kd\\\', \\\'holdout\\\')\\n    }\\n    manifest = pd.concat([wiki_manifest, math_manifest], ignore_index=True, sort=False)\\n    return parts, manifest, provenance\\n\'}\n\n\ndef write_modules(work_dir: Path) -> None:\n    work_dir.mkdir(parents=True, exist_ok=True)\n    for name, source in MODULE_SOURCES.items():\n        # The embedding delimiter contributes one leading newline.\n        source = source[1:] if source.startswith("\\n") else source\n        path = work_dir / name\n        path.write_text(source, encoding="utf-8")\n        print(f"Wrote {path} ({len(source.splitlines())} lines)")\n\n\ndef verify_dependency_environment() -> None:\n    """Preflight the pinned package versions and NumPy/SciPy binary imports.\n\n    A Jupyter kernel can retain old imported Python/C-extension modules after\n    pip updates the files on disk. Check both installed and loaded versions\n    before importing any project modules.\n    """\n    required = {\n        "transformers": "4.57.6",\n        "numpy": "2.2.6",\n        "scipy": "1.16.3",\n    }\n    mismatches = []\n    for package, expected in required.items():\n        try:\n            installed = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            installed = None\n        loaded_module = sys.modules.get(package)\n        loaded = getattr(loaded_module, "__version__", None)\n        print(f"{package}: installed={installed!r}, loaded={loaded!r}, required={expected!r}")\n        if installed != expected:\n            mismatches.append(f"{package} installed={installed!r}, required={expected!r}")\n        if loaded is not None and loaded != expected:\n            mismatches.append(f"{package} loaded={loaded!r}, required={expected!r}")\n\n    repair = (\n        "In a Kaggle notebook cell, run:\\n"\n        "  %pip install --no-cache-dir --force-reinstall "\n        "\'numpy==2.2.6\' \'scipy==1.16.3\' \'transformers==4.57.6\'\\n"\n        "Then use Session > Restart Session and rerun from a fresh kernel. "\n        "Do not install binary packages and continue in the same kernel."\n    )\n    if mismatches:\n        raise RuntimeError("Dependency mismatch: " + "; ".join(mismatches) + "\\n" + repair)\n\n    try:\n        import numpy as _np\n        import scipy as _scipy\n        from scipy.linalg import cho_factor as _cho_factor, cho_solve as _cho_solve\n        x = _np.array([[2.0, 0.0], [0.0, 3.0]])\n        _cho_solve(_cho_factor(x), _np.array([2.0, 3.0]))\n    except Exception as exc:\n        raise RuntimeError(\n            "NumPy/SciPy failed a simple import and Cholesky solve. "\n            "This may indicate mixed compiled/Python package versions.\\n" + repair\n        ) from exc\n    print("NumPy/SciPy import and Cholesky smoke test: PASS")\n\n\ndef run_diagnostics() -> dict:\n    """One concise report for a Kaggle notebook and a fresh Python process.\n\n    This is read-only: it NEVER installs packages, reloads extensions, or runs\n    training. It records the mismatches that matter for this script and groups\n    other pip-check warnings without flooding notebook output.\n    """\n    tracked = {\n        "numpy": "2.2.6",\n        "scipy": "1.16.3",\n        "transformers": "4.57.6",\n        "datasets": "4.4.1",\n        "pandas": "2.2.3",\n        "fsspec": "2025.10.0",\n        "accelerate": "1.12.0",\n    }\n    def version_on_disk(name):\n        try:\n            return importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            return "NOT INSTALLED"\n\n    report = {"packages": {}, "new_process": {}, "pip_check": {}}\n    for name, expected in tracked.items():\n        installed = version_on_disk(name)\n        mod = sys.modules.get(name)\n        loaded = getattr(mod, "__version__", None)\n        report["packages"][name] = {\n            "expected": expected, "installed": installed,\n            "kernel_loaded": loaded,\n            "disk_matches": installed == expected,\n            "kernel_matches": loaded is None or loaded == expected,\n        }\n\n    print("=" * 83)\n    print(" QWEN3 / KAGGLE - SINGLE DEBUG REPORT (nothing is installed or changed)")\n    print("=" * 83)\n    print("%-15s %-14s %-15s %-17s %s" % (\n        "package", "required", "on disk", "loaded in kernel", "status"))\n    for name, p in report["packages"].items():\n        status = "OK" if p["disk_matches"] and p["kernel_matches"] else "MISMATCH"\n        print("%-15s %-14s %-15s %-17s %s" % (\n            name, p["expected"], p["installed"], p["kernel_loaded"] or "(not loaded)", status))\n\n    # Run binary import tests in a new interpreter. Notebook imports can stay\n    # stale even after pip successfully modifies the environment on disk.\n    probe = r"""\nimport json\nimport importlib\nr = {"versions": {}, "smoke_tests": {}, "gpu": None}\nfor name in ("numpy", "scipy", "transformers", "datasets", "fsspec", "pandas"):\n    try:\n        mod = importlib.import_module(name)\n        r["versions"][name] = str(mod.__version__)\n        r["smoke_tests"][name] = "OK"\n    except Exception as e:\n        r["smoke_tests"][name] = "%s: %s" % (type(e).__name__, str(e)[:160])\ntry:\n    import numpy as np\n    from scipy.linalg import cho_factor, cho_solve\n    a = np.array([[2., 0.], [0., 3.]])\n    b = np.array([2., 3.])\n    assert np.allclose(cho_solve(cho_factor(a), b), [1., 1.])\n    r["smoke_tests"]["scipy_cholesky"] = "OK"\nexcept Exception as e:\n    r["smoke_tests"]["scipy_cholesky"] = "%s: %s" % (type(e).__name__, str(e)[:160])\ntry:\n    import torch\n    r["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"\nexcept Exception as e:\n    r["gpu"] = "%s: %s" % (type(e).__name__, str(e)[:140])\nprint("__QWEN3_DIAGNOSTIC_JSON__" + json.dumps(r))\n"""\n    print("\\n[Fresh Python process / actual import checks]")\n    try:\n        result = subprocess.run([sys.executable, "-c", probe], capture_output=True,\n                                text=True, timeout=120, check=False)\n        marker = "__QWEN3_DIAGNOSTIC_JSON__"\n        lines = [line for line in result.stdout.splitlines() if line.startswith(marker)]\n        if not lines:\n            report["new_process"] = {\n                "error": (result.stderr or result.stdout or "No output")[-1000:]}\n            print("  Import subprocess: FAIL -", report["new_process"]["error"][-300:])\n        else:\n            report["new_process"] = json.loads(lines[-1][len(marker):])\n            for name, status in report["new_process"]["smoke_tests"].items():\n                print("  %-20s %s" % (name, status))\n            print("  GPU: %s" % report["new_process"]["gpu"])\n    except Exception as e:\n        report["new_process"] = {"error": "%s: %s" % (type(e).__name__, e)}\n        print("  Import subprocess: FAIL -", report["new_process"]["error"])\n\n    print("\\n[pip check - concise]")\n    try:\n        result = subprocess.run([sys.executable, "-m", "pip", "check"],\n                                capture_output=True, text=True, timeout=45,\n                                check=False)\n        lines = [l.strip() for l in result.stdout.splitlines() if l.strip()]\n        relevant = tuple(tracked) + (\n            "huggingface", "tokenizers", "torch", "s3fs", "gcsfs", "pyarrow")\n        important = [l for l in lines if any(k in l.lower() for k in relevant)]\n        unrelated = [l for l in lines if l not in important]\n        report["pip_check"] = {"exit_code": result.returncode,\n                               "relevant": important, "other": unrelated,\n                               "stderr": result.stderr.strip()[-300:]}\n        if not lines and result.returncode == 0:\n            print("  PASS (no dependency conflicts)")\n        else:\n            for line in important[:8]:\n                print("  Relevant: " + line[:230])\n            if len(important) > 8:\n                print("  ... and %d more relevant conflicts" % (len(important) - 8))\n            print("  Other environment warnings: %d" % len(unrelated))\n            if unrelated:\n                print("  (e.g. %s)" % unrelated[0][:170])\n    except Exception as e:\n        report["pip_check"] = {"error": "%s: %s" % (type(e).__name__, e)}\n        print("  pip check unavailable:", report["pip_check"]["error"])\n\n    problems = [name for name, p in report["packages"].items()\n                if not p["disk_matches"]]\n    stale = [name for name, p in report["packages"].items()\n             if not p["kernel_matches"]]\n    failures = [name for name, value in report["new_process"].get(\n                "smoke_tests", {}).items() if value != "OK"]\n    print("\\n[Conclusion / Next action]")\n    if problems:\n        print("  DISK MISMATCH:", ", ".join(problems))\n        print("  In ONE Kaggle cell, install compatible versions, then Restart Session:")\n        print("  %pip install -q \'numpy==2.2.6\' \'scipy==1.16.3\' \'transformers==4.57.6\' \'datasets==4.4.1\' \'pandas==2.2.3\' \'fsspec==2025.10.0\' \'accelerate==1.12.0\'")\n    elif failures or "error" in report["new_process"]:\n        print("  FRESH PROCESS IMPORT FAIL: check the test errors above. Reinstall only the failing dependencies, then restart.")\n    elif stale:\n        print("  DISK OK, KERNEL STALE:", ", ".join(stale))\n        print("  Restart Session (not just rerun cells), or run the main script with !python in a fresh process.")\n    else:\n        print("  CORE ENVIRONMENT OK. Next: run --checks-only before full training.")\n    dest = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()\n    try:\n        outfile = dest / "qwen3_debug_report.json"\n        outfile.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n        print("  Machine-readable report:", outfile)\n    except Exception as e:\n        print("  Could not save JSON report:", str(e)[:100])\n    return report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    # Jupyter/Colab kernels inject "-f /path/to/kernel.json" into sys.argv.\n    # Accept this internal argument when the script is executed through exec()\n    # or %run without weakening validation of regular command-line flags.\n    parser.add_argument("-f", dest="_jupyter_connection_file",\n                        default=None, help=argparse.SUPPRESS)\n    parser.add_argument("--diagnose", "--debug", action="store_true",\n                        help="Print one compact package/kernel/import/dependency report and exit")\n    parser.add_argument("--mode", choices=("pilot", "quick", "full"), default="pilot")\n    parser.add_argument("--max-merges", type=int, default=None,\n                        help="Compression stages for greedy search (default: 2 quick / 5 full)")\n    parser.add_argument("--pairs", nargs="+", metavar="LEFT-RIGHT",\n                        help="Pilot pairs in one-based original layer numbers")\n    parser.add_argument("--third-pair-candidates", nargs="+", metavar="LEFT-RIGHT",\n                        help="Train these optional third-pair candidates from the fixed stage 2")\n    parser.add_argument("--max-third-ppl-degradation-pct", type=float, default=5.0,\n                        help="Maximum Wiki-40B PPL increase versus Original for the third merge")\n    parser.add_argument("--model-id", default="Qwen/Qwen3-0.6B",\n                        help="Transformers checkpoint compatible with the Qwen3 decoder")\n    parser.add_argument("--train-epochs", type=int, help="Override the preset epoch limit")\n    parser.add_argument("--math-train-fraction", type=float,\n                        help="Fraction of train and holdout windows from MathOctopus")\n    parser.add_argument("--math-train-languages", nargs="+", metavar="LANG")\n    parser.add_argument("--math-max-window-tokens", type=int)\n    parser.add_argument("--eval-languages", nargs="+", choices=("de", "es", "fr", "ja", "zh"))\n    parser.add_argument("--eval-questions", type=int, help="Question cap per benchmark and language")\n    parser.add_argument("--eval-seed", type=int, default=43, help="Seed for the fresh pilot sample")\n    parser.add_argument("--saved-run-dir", type=Path, help="Evaluate an existing locked pilot checkpoint")\n    parser.add_argument("--audit-run-dir", type=Path, help="Audit and exclude an additional prior pilot run")\n    parser.add_argument("--eval-shots", type=int, help="Demonstration cap per question")\n    parser.add_argument("--max-new-tokens", type=int, help="MGSM generation cap")\n    parser.add_argument("--no-mcq-generation", action="store_true",\n                        help="Skip optional MCQ free-generation diagnostic; keep forced-choice scoring")\n    parser.add_argument("--mcq-max-new-tokens", type=int, default=64,\n                        help="Maximum tokens for each optional MCQ free-generated answer")\n    parser.add_argument("--max-prompt-tokens", type=int, help="Maximum tokens per test prompt")\n    parser.add_argument("--token-budget", type=int, help="Input token cap per benchmark and language")\n    parser.add_argument("--work-dir", type=Path, help="Directory for extracted modules and results")\n    parser.add_argument("--install-deps", action="store_true",\n                        help="Install the notebook\'s pinned dependencies first")\n    parser.add_argument("--modules-only", action="store_true",\n                        help="Extract the embedded modules without running")\n    parser.add_argument("--checks-only", action="store_true",\n                        help="Extract modules and run tiny structural self-checks")\n    parser.add_argument("--skip-checks", action="store_true")\n    parser.add_argument("--skip-eval", action="store_true",\n                        help="Train on Wiki-40B but skip held-out benchmarks")\n    args = parser.parse_args()\n    if args.diagnose:\n        run_diagnostics()\n        return\n    started = time.perf_counter()\n    pilot = args.mode == "pilot"\n    if args.saved_run_dir and (not pilot or args.skip_eval):\n        parser.error("--saved-run-dir requires pilot benchmark evaluation")\n    if args.pairs and not pilot:\n        parser.error("--pairs requires --mode pilot")\n    if args.max_merges is not None and (pilot or args.max_merges < 1):\n        parser.error("Use --pairs in pilot mode or a positive --max-merges in quick/full mode")\n    for option in ("train_epochs", "eval_questions", "eval_shots", "max_new_tokens",\n                   "max_prompt_tokens", "token_budget", "mcq_max_new_tokens"):\n        value = getattr(args, option)\n        minimum = 128 if option == "max_prompt_tokens" else 1\n        if value is not None and value < minimum:\n            parser.error(f"--{option.replace(\'_\', \'-\')} must be at least {minimum}")\n    if args.eval_languages and len(set(args.eval_languages)) != len(args.eval_languages):\n        parser.error("--eval-languages must contain unique languages")\n    overrides = {}\n    def parse_pairs(entries):\n        try:\n            pairs = tuple(tuple(map(int, item.split("-"))) for item in entries)\n            if any(len(pair) != 2 for pair in pairs):\n                raise ValueError\n        except ValueError:\n            parser.error("Use separate pair strings such as 7-8 13-14")\n        return pairs\n\n    if args.pairs:\n        pairs = parse_pairs(args.pairs)\n        overrides.update(fixed_layer_pairs=pairs, max_merges=len(pairs))\n    if args.third_pair_candidates:\n        if not pilot or args.saved_run_dir:\n            parser.error("Third-pair candidates require training a new pilot")\n        overrides.update(third_pair_candidates=parse_pairs(args.third_pair_candidates),\n                         max_third_ppl_degradation_pct=args.max_third_ppl_degradation_pct)\n    if not pilot:\n        overrides["max_merges"] = args.max_merges or (5 if args.mode == "full" else 2)\n    if args.train_epochs is not None:\n        overrides["epochs"] = args.train_epochs\n    math_overrides = {key: getattr(args, key) for key in (\n        "math_train_fraction", "math_train_languages", "math_max_window_tokens") if getattr(args, key) is not None}\n    if args.saved_run_dir and math_overrides:\n        parser.error("Math data settings apply when training new merged weights")\n    overrides.update(math_overrides)\n\n    base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()\n    work_dir = (args.work_dir or (base / "hiroyoshi_qwen3_mathoctopus_pilot" if pilot else base)).resolve()\n    write_modules(work_dir)\n    status_path = work_dir / "latest_run.json"\n    if pilot and not (args.modules_only or args.checks_only):\n        status_path.unlink(missing_ok=True)\n    if args.modules_only:\n        return\n    if args.install_deps:\n        if "ipykernel" in sys.modules:\n            raise RuntimeError(\n                "Do not use --install-deps inside a running notebook kernel. "\n                "Run `%pip install --no-cache-dir --force-reinstall "\n                "\'numpy==2.2.6\' \'scipy==1.16.3\' \'transformers==4.57.6\'`, "\n                "then use Session > Restart Session. Alternatively run "\n                "`!python <this_script.py> --install-deps` as a new process."\n            )\n        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",\n                               "--disable-pip-version-check", *DEPENDENCIES])\n\n    # Validate before the modules import transformers. A notebook kernel can\n    # retain an older transformers in sys.modules after pip upgrades it.\n    verify_dependency_environment()\n\n    sys.path.insert(0, str(work_dir))\n    import qwen3_affine_core as sm\n    import evaluation_protocol as ep\n    import qwen3_compression_multi as nc\n    import qwen3_benchmarks as be\n    import verify_qwen3\n\n    if not args.skip_checks:\n        checks = verify_qwen3.run_checks()\n        print(json.dumps(checks, indent=2))\n        assert all(item["status"] == "PASS" for item in checks)\n        (work_dir / "qwen3_self_checks.json").write_text(\n            json.dumps(checks, indent=2), encoding="utf-8")\n    if args.checks_only:\n        return\n\n    import torch\n    if args.saved_run_dir:\n        training_dir = args.saved_run_dir.resolve()\n        cfg = nc.Config(**json.loads((training_dir / "configuration.json").read_text(encoding="utf-8")))\n    else:\n        cfg = nc.make_config(\n            args.mode, model_id=args.model_id,\n            output_dir=str(work_dir / "results") if pilot else nc.Config().output_dir,\n            **overrides,\n        )\n    try:\n        nc.validate_config(cfg)\n    except ValueError as exc:\n        parser.error(str(exc))\n    if not torch.cuda.is_available():\n        raise RuntimeError("GPU required: enable Kaggle GPU accelerator")\n    print("GPU:", torch.cuda.get_device_name(0))\n    print(json.dumps(asdict(cfg), indent=2))\n\n    if args.saved_run_dir:\n        from datetime import datetime, timezone\n\n        print("Reusing locked pilot checkpoints:", training_dir, flush=True)\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n        evaluation_dir = work_dir / "saved_evaluations" / stamp\n        evaluation_dir.mkdir(parents=True, exist_ok=False)\n        methods = ("NonlinearMultiply",)\n        lock = json.loads((training_dir / "selection_locked.json").read_text(encoding="utf-8"))\n        final_stage = int(lock.get("max_merges", cfg.max_merges))\n    else:\n        results = nc.run_experiment(cfg)\n        training_dir = Path(results["output_dir"])\n        evaluation_dir = training_dir / "fewshot_benchmarks"\n        methods = results["methods"]\n        final_stage = results["final_stage"]\n        print("Saved to:", training_dir)\n        print("Wiki-40B validation baseline:", results["baseline"])\n        print(results["selected"][[\n            "Compression_Steps", "Method", "Plan_1based", "Validation_PPL",\n            "PPL_Degradation_%", "Remaining_Blocks", "Parameter_Reduction_%", "Best_Epoch",\n        ]].to_string(index=False))\n        print(results["pivot"].to_string())\n\n    status = {\n        "output_dir": str(training_dir), "evaluation_dir": str(evaluation_dir), "model_id": cfg.model_id,\n        "methods": methods, "fixed_layer_pairs_1based": cfg.fixed_layer_pairs,\n        "third_pair_candidates_1based": cfg.third_pair_candidates, "final_stage": final_stage,\n        "training_complete": True, "training_reused": bool(args.saved_run_dir),\n        "evaluation_complete": False, "evaluation_skipped": args.skip_eval,\n        "eval_seed": args.eval_seed if pilot else cfg.seed,\n        "training_elapsed_seconds": None if args.saved_run_dir else round(time.perf_counter() - started, 1),\n    }\n    if pilot:\n        ep._write_json(status_path, status)\n    else:\n        backup = shutil.make_archive(\n            str(training_dir) + "_PRETEST_BACKUP", "zip", root_dir=training_dir)\n        print("Training backup:", backup)\n\n    if not args.skip_eval:\n        max_questions = args.eval_questions or (12 if pilot else None)\n        max_prompt_tokens = args.max_prompt_tokens or (1024 if pilot else 3072)\n        languages = tuple(args.eval_languages or (("de", "ja") if pilot else ("de", "es", "fr", "ja", "zh")))\n        shots = args.eval_shots or (3 if pilot else None)\n        prepare = None\n        if pilot:\n            import pandas as pd\n            import fresh_evaluation_protocol as fresh\n\n            known_run = (base / "hiroyoshi_qwen3_layer4_pilot" / "results"\n                         / "multi_merge_pilot_20261107T111356_048699Z")\n            prior_runs = list(dict.fromkeys(p.resolve() for p in (args.audit_run_dir, args.saved_run_dir)\n                                            if p is not None))\n            if not prior_runs and (known_run / "fewshot_benchmarks" / "fewshot_predictions.csv").is_file():\n                prior_runs.append(known_run)\n            archived = []\n            for prior in prior_runs:\n                prior_csv = prior / "fewshot_benchmarks" / "fewshot_predictions.csv"\n                if prior_csv.is_file():\n                    frame = pd.read_csv(prior_csv, keep_default_na=False)\n                    frame["Archived_Run"] = str(prior)\n                    archived.append(frame)\n                elif args.audit_run_dir is not None and prior == args.audit_run_dir.resolve():\n                    raise FileNotFoundError(f"Cannot find archived predictions in {prior}")\n            previous = pd.concat(archived, ignore_index=True) if archived else pd.DataFrame()\n            evaluation_dir.mkdir(parents=True, exist_ok=True)\n            if archived:\n                fresh.audit_mgsm(previous, evaluation_dir)\n            else:\n                print("Prior completion audit requires a saved predictions CSV. Historical question exclusions still apply.",\n                      flush=True)\n\n            def prepare(tokenizer, destination):\n                return fresh.prepare_fresh(\n                    tokenizer, destination, previous, args.eval_seed, max_questions, max_prompt_tokens,\n                    languages=languages, shots=shots, token_budget=args.token_budget,\n                )\n\n        scores, budgets = be.run_fewshot_evaluation(\n            training_dir, languages=languages,\n            token_budget=args.token_budget or (max_questions * max_prompt_tokens if pilot else 100_000),\n            stages=(final_stage,) if pilot else tuple(range(1, final_stage + 1)),\n            methods=methods, max_new_tokens=args.max_new_tokens or 128,\n            max_questions=max_questions, max_shots=shots, max_prompt_tokens=max_prompt_tokens,\n            mcq_generation=not args.no_mcq_generation, mcq_max_new_tokens=args.mcq_max_new_tokens,\n            evaluation_dir=evaluation_dir, prepare=prepare,\n        )\n        if pilot:\n            fresh.verify_paired_predictions(evaluation_dir)\n            ep._write_json(evaluation_dir / "fresh_evaluation.json", status | {"evaluation_complete": True})\n            status["evaluation_complete"] = True\n            ep._write_json(status_path, status)\n        print(budgets.to_string(index=False))\n        print(scores.sort_values(["Benchmark", "Language", "Compression_Steps", "Method"]).to_string(index=False))\n\n    if args.saved_run_dir:\n        archive = shutil.make_archive(str(evaluation_dir) + "_FRESH_EVALUATION", "zip", root_dir=evaluation_dir)\n    else:\n        for name in list(MODULE_SOURCES) + ["qwen3_self_checks.json"]:\n            src = work_dir / name\n            if src.exists():\n                shutil.copy2(src, training_dir / name)\n        archive = shutil.make_archive(str(training_dir) + "_ALL_RESULTS", "zip", root_dir=training_dir)\n    if pilot:\n        status.update(results_zip=archive, elapsed_seconds=round(time.perf_counter() - started, 1))\n        ep._write_json(status_path, status)\n    print("Results ZIP:", archive)\n\n\nif __name__ == "__main__":\n    main()\n', 'launch_hiroyoshi.py': '"""Reuse the existing isolated Kaggle environment; do not change global packages."""\nimport os, shutil, subprocess, sys\nfrom pathlib import Path\nos.environ.setdefault(\'USE_TF\', \'0\')\nos.environ.setdefault(\'USE_FLAX\', \'0\')\nCORE_REQUIREMENTS = (\'transformers==4.57.6\', \'numpy==2.2.6\', \'scipy==1.16.3\')\nOPTIONAL_REQUIREMENTS = (\'datasets==4.4.1\', \'pandas==2.2.3\', \'accelerate==1.12.0\', \'threadpoolctl==3.6.0\', \'fsspec==2025.10.0\')\n\ndef _venv_python(env_dir: Path) -> Path:\n    return env_dir / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")\n\ndef _bootstrap_env() -> Path:\n    """Create a Kaggle-friendly venv without running ensurepip.\n\n    Kaggle\'s Python 3.13 environment may fail in venv.ensurepip even though\n    system pip is already installed. Sharing system site-packages makes that\n    existing pip (and GPU-enabled torch) accessible to the child interpreter.\n    Core pins are installed *inside the venv* and do not modify Kaggle\'s base.\n    """\n    import os\n    import venv\n    root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()\n    env_dir = root / ".qwen3_env"\n    py = _venv_python(env_dir)\n    log_path = root / "qwen3_setup.log"\n    clean_env = os.environ.copy()\n    clean_env.pop("PYTHONPATH", None)\n    clean_env["PYTHONNOUSERSITE"] = "1"\n\n    # Disable pip bootstrapping: Kaggle can have ensurepip disabled or broken.\n    # We inherit base-system pip through system_site_packages=True instead.\n    if not py.is_file():\n        print("[setup] Creating Qwen3 experiment environment without ensurepip...", flush=True)\n        try:\n            venv.EnvBuilder(with_pip=False, system_site_packages=True).create(env_dir)\n        except Exception as exc:\n            raise RuntimeError(\n                f"Cannot create the dedicated venv at {env_dir}: {exc}"\n            ) from exc\n\n    # Check the child interpreter itself first, including pip availability.\n    # An earlier failed setup can leave a half-created .qwen3_env behind.\n    pip_probe = subprocess.run(\n        [str(py), "-m", "pip", "--version"], capture_output=True, text=True,\n        env=clean_env, timeout=60,\n    )\n    if pip_probe.returncode:\n        print("[setup] Repairing incomplete Qwen3 experiment environment...", flush=True)\n        # This is a script-owned directory, never a Kaggle system directory.\n        shutil.rmtree(env_dir, ignore_errors=True)\n        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(env_dir)\n        pip_probe = subprocess.run(\n            [str(py), "-m", "pip", "--version"], capture_output=True, text=True,\n            env=clean_env, timeout=60,\n        )\n        if pip_probe.returncode:\n            raise RuntimeError(\n                "Python\'s base pip is not available inside the venv. "\n                "Cannot install dependencies safely. "\n                "pip diagnostics:\\n" + (pip_probe.stdout + pip_probe.stderr)[-2500:]\n            )\n\n    probe = r"""\nimport importlib.metadata as m\nimport numpy, scipy, transformers\nfrom scipy.linalg import cho_factor, cho_solve\nimport datasets, pandas, accelerate, torch, threadpoolctl, fsspec\nfor package, version in [(\'datasets\',\'4.4.1\'), (\'pandas\',\'2.2.3\'), (\'accelerate\',\'1.12.0\'), (\'threadpoolctl\',\'3.6.0\'), (\'fsspec\',\'2025.10.0\')]:\n    assert m.version(package) == version, (package, m.version(package), version)\nfrom transformers import Qwen3Config, Qwen3ForCausalLM\nassert m.version(\'numpy\')==\'2.2.6\'\nassert m.version(\'scipy\')==\'1.16.3\'\nassert m.version(\'transformers\')==\'4.57.6\'\nimport numpy as np\nx = np.array([[2.,0.],[0.,3.]])\nassert np.allclose(cho_solve(cho_factor(x), np.array([2.,3.])), [1.,1.])\n"""\n    check = subprocess.run(\n        [str(py), "-c", probe], capture_output=True, text=True,\n        env=clean_env, timeout=180,\n    )\n    if check.returncode == 0:\n        print("[setup] Qwen3 experiment environment: READY", flush=True)\n        return py\n\n    print("[setup] Installing Qwen3 core packages into the venv only...", flush=True)\n    requirements = [*CORE_REQUIREMENTS, *OPTIONAL_REQUIREMENTS]\n    command = [\n        str(py), "-m", "pip", "install", "--disable-pip-version-check",\n        "--no-input", "--no-cache-dir", *requirements,\n    ]\n    with log_path.open("w", encoding="utf-8") as log:\n        log.write("Command: " + " ".join(command) + "\\n\\n")\n        log.write("Initial import probe:\\n" + (check.stdout + check.stderr)[-2000:] + "\\n\\n")\n        log.flush()\n        installed = subprocess.run(\n            command, stdout=log, stderr=subprocess.STDOUT, env=clean_env,\n        )\n    if installed.returncode:\n        tail = log_path.read_text(errors="replace")[-4500:]\n        raise RuntimeError(\n            f"Qwen3 dependency installation failed (pip exit {installed.returncode}). "\n            f"Check Kaggle Internet and available disk space. Log: {log_path}\\n{tail}"\n        )\n    check = subprocess.run(\n        [str(py), "-c", probe], capture_output=True, text=True,\n        env=clean_env, timeout=180,\n    )\n    if check.returncode:\n        raise RuntimeError(\n            "Dependencies installed, but Qwen3 imports failed:\\n" +\n            (check.stdout + check.stderr)[-4500:] + "\\nSetup log: " + str(log_path)\n        )\n    print("[setup] Qwen3 experiment environment: READY", flush=True)\n    return py\n\nif __name__ == \'__main__\':\n    py = _bootstrap_env()\n    env = os.environ.copy()\n    env.pop(\'PYTHONPATH\', None)\n    env[\'PYTHONNOUSERSITE\'] = \'1\'\n    command = [str(py), \'-u\', str(Path(__file__).with_name(\'Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py\')), *sys.argv[1:]]\n    raise SystemExit(subprocess.call(command, env=env))\n'}
EXPECTED_SHA256 = {'Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py': '5842f725abbbfafa460c63cad8e27da61d50b68876c8f273da719ecc70ee1089', 'launch_hiroyoshi.py': '42a0f83ab56d1dda1e1a6643fc1b0bf512a3124757a1391a6bb94a04b5bbdbf6'}

for name, source in SOURCES.items():
    payload = source.encode('utf-8')
    if sha256(payload).hexdigest() != EXPECTED_SHA256[name]:
        raise ValueError(f'Embedded source checksum mismatch: {name}')
    compile(source, name, 'exec')
    (WORK / name).write_bytes(payload)
print('Pilot script SHA-256:', EXPECTED_SHA256['Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py'])

Pilot script SHA-256: 5842f725abbbfafa460c63cad8e27da61d50b68876c8f273da719ecc70ee1089


## Quick answer parser tests

Run this cell after exporting the source. The checks cover multilingual MCQ answers, conflicting labels, MGSM number formats, and truncated responses.

In [9]:
import ast
import re
import unicodedata

source = (WORK / 'Hiroyoshi_Qwen3_Fixed_Pair_Pilot.py').read_text(encoding='utf-8')
outer_tree = ast.parse(source)
modules_assignment = next(node for node in outer_tree.body if isinstance(node, ast.Assign)
                          and any(isinstance(t, ast.Name) and t.id == 'MODULE_SOURCES'
                                  for t in node.targets))
embedded_modules = ast.literal_eval(modules_assignment.value)
benchmark_tree = ast.parse(embedded_modules['qwen3_benchmarks.py'])
parser_node = next(node for node in benchmark_tree.body if isinstance(node, ast.FunctionDef)
                   and node.name == 'parse_mcq_generated_answer')
namespace = {'re': re, 'unicodedata': unicodedata}
exec(compile(ast.Module(body=[parser_node], type_ignores=[]), '<MCQ parser test>', 'exec'), namespace)
parse_answer = namespace['parse_mcq_generated_answer']

examples = [
    ('C', 'C'),
    ('C. Rückgewinnbare fossile Brennstoffe', 'C'),
    ('The correct answer is:\n\nC. Rückgewinnbare fossile Brennstoffe', 'C'),
    ('答案是：\n\nC', 'C'),
    ('正解はＣです', 'C'),
    ('**B**', 'B'),
    ('Respuesta final: D', 'D'),
    ('The options A and B were considered, but no final answer was provided.', None),
    ('Final answer: A\nFinal answer: B', None),
    ('A. first\nB. second', None),
]
for completion, expected in examples:
    actual = parse_answer(completion)
    assert actual == expected, f'Expected {expected!r}, got {actual!r}: {completion!r}'
print(f'PASS: all {len(examples)} MCQ answer-recognition examples were handled correctly.')

strict = {}
exec(compile(embedded_modules['strict_mgsm.py'], '<MGSM parser test>', 'exec'), strict)
classify = strict['classify']
assert classify('Final answer: 1,5', 'de', '1.5', 'eos')[-1] == 'correct'
assert classify('I tried 9 and then stopped.', 'de', '9', 'eos')[-1] == 'missing_final_answer'
assert classify('Final answer: 9', 'ja', '9', 'max_new_tokens')[-1] == 'truncated'
print('PASS: decimal commas, missing final answers, and truncation.')

PASS: all 10 MCQ answer-recognition examples were handled correctly.
PASS: decimal commas, missing final answers, and truncation.


In [10]:
import os
import signal
import subprocess
import sys
import time

command = [
    sys.executable, '-u', str(WORK / 'launch_hiroyoshi.py'),
    '--mode', 'pilot', '--work-dir', str(WORK), '--model-id', MODEL_ID,
    '--pairs', *FIXED_PAIRS, '--train-epochs', str(TRAIN_EPOCHS),
    '--eval-languages', *EVAL_LANGUAGES,
    '--eval-questions', str(QUESTIONS_PER_BENCHMARK_LANGUAGE), '--eval-seed', str(EVAL_SEED),
    '--eval-shots', str(DEMONSTRATIONS_PER_QUESTION),
    '--max-prompt-tokens', str(MAX_PROMPT_TOKENS),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--mcq-max-new-tokens', str(MCQ_GENERATION_MAX_NEW_TOKENS),
]
if SAVED_RUN_DIR is not None:
    command.extend(('--saved-run-dir', str(Path(SAVED_RUN_DIR).resolve())))
else:
    command.extend(('--math-train-fraction', str(MATH_TRAIN_FRACTION),
                    '--math-train-languages', *MATH_TRAIN_LANGUAGES,
                    '--math-max-window-tokens', str(MATH_MAX_WINDOW_TOKENS)))
    if THIRD_PAIR_CANDIDATES:
        command.extend(('--third-pair-candidates', *THIRD_PAIR_CANDIDATES,
                        '--max-third-ppl-degradation-pct', str(MAX_THIRD_PPL_DEGRADATION_PCT)))
if AUDIT_RUN_DIR is not None:
    command.extend(('--audit-run-dir', str(Path(AUDIT_RUN_DIR).resolve())))
if not RUN_MCQ_GENERATION_DIAGNOSTIC:
    command.append('--no-mcq-generation')
if TIME_LIMIT_SECONDS is not None and (os.name != 'posix' or TIME_LIMIT_SECONDS <= 0):
    raise ValueError('A positive time limit is supported on Kaggle/Linux.')

(WORK / 'latest_run.json').unlink(missing_ok=True)
log = WORK / 'pilot.log'
print('Pilot log:', log, flush=True)
started = time.perf_counter()
with log.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen(command, cwd=WORK, stdout=stream,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        code = process.wait(timeout=TIME_LIMIT_SECONDS)
    except (subprocess.TimeoutExpired, KeyboardInterrupt) as exc:
        if os.name == 'posix':
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
        else:
            process.kill()
        process.wait()
        if isinstance(exc, subprocess.TimeoutExpired):
            raise TimeoutError(f'Pilot stopped after {TIME_LIMIT_SECONDS}s. See {log}.') from exc
        raise
print(log.read_text(encoding='utf-8', errors='replace')[-24000:])
if code:
    raise RuntimeError(f'Pilot failed (exit {code}). See {log} and {BASE / "qwen3_setup.log"}.')
print(f'Notebook run including setup: {time.perf_counter() - started:.1f}s')

Pilot log: /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/pilot.log
[setup] Installing Qwen3 core packages into the venv only...
[setup] Qwen3 experiment environment: READY
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/qwen3_affine_core.py (192 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/evaluation_protocol.py (148 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/qwen3_compression_multi.py (1143 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/qwen3_benchmarks.py (532 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/verify_qwen3.py (74 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/strict_mgsm.py (83 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/fresh_evaluation_protocol.py (101 lines)
Wrote /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/mathoctopus_training.py (150 lines)
transformers: installed='4.57.6', loaded=None, required='4.57

In [11]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

status_path = WORK / 'latest_run.json'
if not status_path.exists():
    raise RuntimeError(f'Training summary is unavailable. See {WORK / "pilot.log"}.')
status = json.loads(status_path.read_text(encoding='utf-8'))
run_dir = Path(status['output_dir'])
baseline = json.loads((run_dir / 'baseline_validation.json').read_text(encoding='utf-8'))
print('Original Wiki-40B validation perplexity:', baseline['PPL'])
print('Fixed original layer pairs:', status['fixed_layer_pairs_1based'])
print('Selected merge count:', status['final_stage'])
source_mix = run_dir / 'train_source_mix.csv'
if source_mix.exists():
    print('Training and holdout data by source')
    display(pd.read_csv(source_mix))
selected = pd.read_csv(run_dir / 'selected_by_stage.csv')
display(selected[['Compression_Steps', 'Plan_1based', 'Remaining_Blocks',
                  'Validation_PPL', 'PPL_Degradation_%', 'Parameter_Reduction_%', 'Best_Epoch']])
candidate_path = run_dir / 'all_candidates.csv'
if candidate_path.exists():
    candidates = pd.read_csv(candidate_path)
    if 'Optional_Stage' in candidates.columns:
        optional = candidates.loc[candidates['Optional_Stage']]
        if not optional.empty:
            print('Third-pair candidates on Wiki-40B validation')
            display(optional[['Candidate_Pair_1based', 'Validation_PPL', 'PPL_Degradation_%',
                              'Within_Optional_PPL_Budget', 'Best_Epoch']])
if status['evaluation_complete']:
    evaluation_dir = Path(status['evaluation_dir'])
    budget = pd.read_csv(evaluation_dir / 'prompt_budget.csv')
    print('Fresh questions and overlap check')
    display(budget[['Benchmark', 'Language', 'Questions', 'Verified_Overlap']])
    print('MGSM outcomes')
    display(pd.read_csv(evaluation_dir / 'mgsm_outcomes.csv'))
    audit_path = evaluation_dir / 'prior_mgsm_audit.csv'
    if audit_path.exists():
        audit = pd.read_csv(audit_path)
        print('Prior MGSM audit')
        display(audit.groupby(['Language', 'Method'], as_index=False).agg(
            Questions=('Source_Index', 'size'), Prior_Correct=('Prior_Correct', 'sum'),
            Strict_Text_Correct=('Strict_Text_Correct', 'sum'), Scoring_Changed=('Scoring_Changed', 'sum'),
        ))
    accuracy = pd.read_csv(evaluation_dir / 'fewshot_accuracy.csv')
    display(accuracy[['Benchmark', 'Language', 'Method', 'Correct', 'Questions',
                      'Accuracy', 'Accuracy_Delta_pp']])
    generated_path = evaluation_dir / 'fewshot_generated_accuracy.csv'
    if generated_path.exists():
        print('Separate MCQ free-generation diagnostic (corrected multilingual answer extraction)')
        generated_accuracy = pd.read_csv(generated_path)
        display(generated_accuracy[['Benchmark', 'Language', 'Method', 'Generated_Correct',
                                    'Generated_Valid', 'Questions', 'Generated_Accuracy',
                                    'Generated_Valid_Rate']])
    choices = pd.read_csv(evaluation_dir / 'fewshot_choice_counts.csv')
    print('Multiple-choice distribution and median probability assigned to A–D')
    display(choices.round({'Median_Choice_Probability_pct': 3}))
else:
    print('Benchmark evaluation incomplete. See the pilot log.')
print('Output directory:', run_dir)
if status.get('results_zip'):
    print('Results ZIP:', status['results_zip'])

Original Wiki-40B validation perplexity: 11.083639818499579
Fixed original layer pairs: [[7, 8], [13, 14]]
Selected merge count: 2
Training and holdout data by source


,Split,Dataset,Windows,Tokens
0,holdout,Mathoctopus/GSM8KInstruct_Parallel,4,808
1,holdout,google/wiki40b,4,512
2,kd,Mathoctopus/GSM8KInstruct_Parallel,8,1621
3,kd,google/wiki40b,8,1024
4,local,Mathoctopus/GSM8KInstruct_Parallel,8,1684
5,local,google/wiki40b,8,1024


,Compression_Steps,Plan_1based,Remaining_Blocks,Validation_PPL,PPL_Degradation_%,Parameter_Reduction_%,Best_Epoch
0,1,7-8,27,11.326652,2.192534,2.639199,10
1,2,7-8;13-14,26,11.472308,3.506679,5.278398,3


Fresh questions and overlap check


,Benchmark,Language,Questions,Verified_Overlap
0,MMMLU,de,40,0
1,MMMLU,ja,40,0
2,MMMLU,fr,40,0
3,INCLUDE,de,40,0
4,INCLUDE,ja,40,0
5,INCLUDE,fr,40,0
6,MGSM,de,40,0
7,MGSM,ja,40,0
8,MGSM,fr,40,0


MGSM outcomes


,Language,Method,MGSM_Outcome,Questions
0,de,NonlinearMultiply,wrong_answer,40
1,de,Original,correct,10
2,de,Original,missing_final_answer,1
3,de,Original,wrong_answer,29
4,fr,NonlinearMultiply,wrong_answer,40
5,fr,Original,correct,10
6,fr,Original,truncated,1
7,fr,Original,wrong_answer,29
8,ja,NonlinearMultiply,wrong_answer,40
9,ja,Original,correct,3


,Benchmark,Language,Method,Correct,Questions,Accuracy,Accuracy_Delta_pp
0,INCLUDE,de,NonlinearMultiply,13,40,0.325,-2.5
1,INCLUDE,de,Original,14,40,0.350,0.0
2,INCLUDE,fr,NonlinearMultiply,12,40,0.300,0.0
3,INCLUDE,fr,Original,12,40,0.300,0.0
4,INCLUDE,ja,NonlinearMultiply,5,40,0.125,-17.5
5,INCLUDE,ja,Original,12,40,0.300,0.0
6,MGSM,de,NonlinearMultiply,0,40,0.000,-25.0
7,MGSM,de,Original,10,40,0.250,0.0
8,MGSM,fr,NonlinearMultiply,0,40,0.000,-25.0
9,MGSM,fr,Original,10,40,0.250,0.0


Multiple-choice distribution and median probability assigned to A–D


,Benchmark,Language,Method,Compression_Steps,Questions,A,B,C,D,Median_Choice_Probability_pct,All_Same_Letter
0,INCLUDE,de,NonlinearMultiply,2,40,24,11,3,2,99.906,False
1,INCLUDE,de,Original,0,40,9,14,10,7,99.987,False
2,INCLUDE,fr,NonlinearMultiply,2,40,9,6,16,9,99.940,False
3,INCLUDE,fr,Original,0,40,3,6,20,11,99.995,False
4,INCLUDE,ja,NonlinearMultiply,2,40,19,6,6,9,99.884,False
5,INCLUDE,ja,Original,0,40,9,5,11,15,99.988,False
6,MMMLU,de,NonlinearMultiply,2,40,15,6,6,13,99.758,False
7,MMMLU,de,Original,0,40,7,6,9,18,99.984,False
8,MMMLU,fr,NonlinearMultiply,2,40,18,5,9,8,99.820,False
9,MMMLU,fr,Original,0,40,6,9,10,15,99.986,False


Output directory: /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/results/multi_merge_pilot_20261109T102908_622005Z
Results ZIP: /kaggle/working/hiroyoshi_qwen3_mathoctopus_supervised/results/multi_merge_pilot_20261109T102908_622005Z_ALL_RESULTS.zip


In [12]:
end_time=time.time()

print(end_time-start_time)

5622.740763902664


## Answer choices for each test question

This table shows the A–D choices from Original and the final `NonlinearMultiply` model, alongside the correct choice, for MMMLU and INCLUDE. `Question ID` is the zero-based row number in that benchmark's test split.

Run this cell by itself if the original evaluation files are still in the Kaggle session. If you extracted the results ZIP elsewhere, set `PREDICTIONS_CSV` to the path of `fewshot_predictions.csv`. To inspect the exact prompt and options for a question, set `SHOW_PROMPT_FOR` using the benchmark, language, and question ID from the table. This uses the saved prompt file beside the CSV.

In [13]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PREDICTIONS_CSV = None  # Or set a path to fewshot_predictions.csv from the results ZIP.
BENCHMARK = None  # Optional filter, e.g. 'MMMLU' or 'INCLUDE'.
LANGUAGE = None  # Optional filter, e.g. 'de' or 'ja'.
SHOW_ONLY_DIFFERENCES = False
SHOW_PROMPT_FOR = None  # Optional tuple, e.g. ('MMMLU', 'de', 123).

if PREDICTIONS_CSV is None:
    base = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
    status_path = base / 'hiroyoshi_qwen3_mathoctopus_pilot' / 'latest_run.json'
    if not status_path.is_file():
        raise FileNotFoundError('Run the benchmark first or set PREDICTIONS_CSV to the saved predictions.')
    status = json.loads(status_path.read_text(encoding='utf-8'))
    if not status['evaluation_complete']:
        raise RuntimeError('The benchmark evaluation is incomplete. See pilot.log.')
    csv_path = Path(status['evaluation_dir']) / 'fewshot_predictions.csv'
else:
    csv_path = Path(PREDICTIONS_CSV)

predictions = pd.read_csv(csv_path, keep_default_na=False)
mcq = predictions.loc[predictions['Benchmark'].isin(('MMMLU', 'INCLUDE'))]
original = mcq.loc[(mcq['Method'] == 'Original') & (mcq['Compression_Steps'] == 0)]
merged = mcq.loc[mcq['Method'] == 'NonlinearMultiply']
if original.empty or merged.empty:
    raise ValueError('The CSV must contain multiple-choice predictions for Original and NonlinearMultiply.')
final_stage = int(merged['Compression_Steps'].max())
merged = merged.loc[merged['Compression_Steps'] == final_stage]

question_key = ['Benchmark', 'Language', 'Source_Index', 'Prompt_SHA256']
if 'Scoring_Prompt_SHA256' in mcq.columns:
    question_key.append('Scoring_Prompt_SHA256')
generated_columns = (["MCQ_Generated_Prediction", "MCQ_Generated_Valid", "MCQ_Generated_Correct"]
                     if "MCQ_Generated_Prediction" in mcq.columns else [])
columns = question_key + ['Gold', 'Prediction'] + generated_columns
paired = original[columns].merge(
    merged[columns], on=question_key, how='outer', validate='one_to_one',
    suffixes=('_original', '_merged'), indicator=True,
)
if not paired['_merge'].eq('both').all() or not paired['Gold_original'].eq(paired['Gold_merged']).all():
    raise ValueError('Original and merged predictions must have matching prompts and correct choices.')

choices = paired.rename(columns={
    'Source_Index': 'Question ID', 'Gold_original': 'Correct choice',
    'Prediction_original': 'Original choice', 'Prediction_merged': 'Merged choice',
})
choices['Original correct'] = choices['Original choice'].eq(choices['Correct choice'])
choices['Merged correct'] = choices['Merged choice'].eq(choices['Correct choice'])
if BENCHMARK is not None:
    choices = choices.loc[choices['Benchmark'] == BENCHMARK]
if LANGUAGE is not None:
    choices = choices.loc[choices['Language'] == LANGUAGE]
if SHOW_ONLY_DIFFERENCES:
    choices = choices.loc[choices['Original choice'] != choices['Merged choice']]
choices = choices.sort_values(['Benchmark', 'Language', 'Question ID']).reset_index(drop=True)

print(f'{len(choices)} questions shown. Merged model stage: {final_stage}.')
shown = ['Benchmark', 'Language', 'Question ID', 'Correct choice',
         'Original choice', 'Merged choice', 'Original correct', 'Merged correct']
if generated_columns:
    shown += ['MCQ_Generated_Prediction_original', 'MCQ_Generated_Prediction_merged',
              'MCQ_Generated_Valid_original', 'MCQ_Generated_Valid_merged']
with pd.option_context('display.max_rows', None):
    display(choices[shown])

if SHOW_PROMPT_FOR is not None:
    benchmark, language, question_id = SHOW_PROMPT_FOR
    match = paired.loc[(paired['Benchmark'] == benchmark) & (paired['Language'] == language)
                       & (paired['Source_Index'] == question_id)]
    if len(match) != 1:
        raise ValueError('Use a benchmark, language, and question ID from the comparison table.')
    prompt_file = csv_path.parent / f'prompts_{benchmark}_{language}.jsonl'
    with prompt_file.open(encoding='utf-8') as prompts:
        for line in prompts:
            sample = json.loads(line)
            if sample['prompt_sha256'] == match.iloc[0]['Prompt_SHA256']:
                print(sample['prompt'] + sample.get('assistant_prefill', ''))
                break
        else:
            raise ValueError('The saved prompt file does not contain the matching prompt.')

FileNotFoundError: Run the benchmark first or set PREDICTIONS_CSV to the saved predictions.